# ML Intraday V3 — Enhanced Pipeline Runner
Runs and inspects the **ml_intraday_v3** pipeline stages with **comprehensive statistics** and **Monte Carlo simulation**.

**Enhancements:**
- Automatic stats printing after each stage
- Monte Carlo simulation for robustness testing
- Trade sequence randomization tests
- Bootstrap confidence intervals

**Assumptions**
- Run from the repo root (contains `ml_intraday_v3/`).
- Virtual env is activated and deps installed.
- `ml_intraday_v3/configs/data.yaml` points to your raw data.

## 0) Parameters & Imports

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import uuid
import os, json, subprocess, sys
import importlib.util
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Notebook config
%matplotlib inline
sns.set_style('darkgrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RUN_ID = os.environ.get("MLV3_RUN_ID")
if not RUN_ID:
    RUN_ID = "run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
os.environ["MLV3_RUN_ID"] = RUN_ID
REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "ml_intraday_v3").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "ml_intraday_v3").exists():
            REPO_ROOT = parent
            break
    else:
        raise RuntimeError("Repo root not found. Run this notebook from the repo root.")

# Add repo root to Python path so ml_intraday_v3 can be imported
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Also add ml_intraday_v3 directory to path for absolute imports (from core.instrument, etc.)
MLV3_DIR = REPO_ROOT / "ml_intraday_v3"
if str(MLV3_DIR) not in sys.path:
    sys.path.insert(0, str(MLV3_DIR))

PYTHON = sys.executable
CLI_MODULE = "ml_intraday_v3.cli"

CONFIG_DIR = REPO_ROOT / "ml_intraday_v3" / "configs"
DATA_YAML = CONFIG_DIR / "data.yaml"
FEATURES_YAML = CONFIG_DIR / "features.yaml"
LABELING_YAML = CONFIG_DIR / "labeling.yaml"
VALIDATION_YAML = CONFIG_DIR / "validation.yaml"
TRAINING_YAML = CONFIG_DIR / "training.yaml"
BACKTEST_YAML = CONFIG_DIR / "backtest.yaml"
EXPERIMENT_GRID_YAML = CONFIG_DIR / "experiment_grid.yaml"
WALKFORWARD_YAML = CONFIG_DIR / "walkforward.yaml"
EXECUTION_SPEC_YAML = CONFIG_DIR / "execution_spec.yaml"
RISK_YAML = CONFIG_DIR / "risk.yaml"

RUN_DIR = REPO_ROOT / "runs" / RUN_ID
BAR_SIZES = ["1m"]  # Focus on 5m for now
SEED = 42
CV_KIND = "purged_kfold"

HAS_PYARROW = importlib.util.find_spec("pyarrow") is not None

BAR_SIZE_1M_DIR = "bar_size=1m"

print("Repo root:", REPO_ROOT)
print("Run ID:", RUN_ID)
print("Run dir:", RUN_DIR)
print("Python:", PYTHON)
print("pyarrow available:", HAS_PYARROW)

Repo root: /Users/eshaanganguly/Documents/projects/algos 3 topstep
Run ID: run_20251227_042529_c5a31e4b
Run dir: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b
Python: /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python
pyarrow available: True


## 1) Enhanced Helper Functions

In [2]:
import hashlib
import yaml
from ml_intraday_v3.experiments.diagnostics import compute_dsr

def read_config(path):
    path = Path(path)
    with open(path, "r") as f:
        if path.suffix == ".json":
            return json.load(f)
        if path.suffix in {".yaml", ".yml"}:
            return yaml.safe_load(f)
    raise ValueError(f"Unsupported config format: {path}")

def hash_content(content):
    if isinstance(content, dict):
        content_bytes = json.dumps(content, sort_keys=True).encode("utf-8")
    elif isinstance(content, str):
        content_bytes = content.encode("utf-8")
    elif isinstance(content, bytes):
        content_bytes = content
    else:
        raise ValueError(f"Unsupported content type: {type(content)}")
    return hashlib.sha256(content_bytes).hexdigest()

def config_hash(path):
    return hash_content(read_config(path))

def analyze_overfitting(trades_df, bar_size="1m"):
    """Compute selection-bias-aware DSR on per-trade and per-day pnl_usd."""
    print_section(f"OVERFITTING DIAGNOSTICS ({bar_size})")
    
    validation_cfg = read_config(VALIDATION_YAML)
    dsr_cfg = (validation_cfg.get('overfitting_diagnostics', {}) or {}).get('dsr', {}) or {}
    n_trials = int(dsr_cfg.get('n_trials', 10))
    target_sharpe = float(dsr_cfg.get('target_sharpe', 0.0))

    if trades_df is None or trades_df.empty:
        print("No trades_df available")
        return None
    executed = trades_df[trades_df.get('executed', False) == True].copy()
    executed = executed.dropna(subset=['pnl_usd']) if 'pnl_usd' in executed.columns else executed
    if executed.empty:
        print("No executed trades")
        return None

    per_trade = executed['pnl_usd'].astype(float).tolist()
    per_day = []
    if 'exit_ts' in executed.columns:
        exit_ts = pd.to_datetime(executed['exit_ts'], utc=True, errors='coerce')
        daily = (
            executed.assign(_exit_date=exit_ts.dt.floor('D'))
            .dropna(subset=['_exit_date'])
            .groupby('_exit_date')['pnl_usd']
            .sum()
        )
        per_day = daily.astype(float).tolist()

    out = {
        'per_trade': compute_dsr(per_trade, n_trials=n_trials, target_sharpe=target_sharpe),
        'per_day': compute_dsr(per_day, n_trials=n_trials, target_sharpe=target_sharpe) if per_day else {'dsr': None, 'reason': 'no_daily_returns'},
        'n_trials': n_trials,
        'target_sharpe': target_sharpe,
        'n_trades': int(len(per_trade)),
        'n_days': int(len(per_day)),
    }

    print("DSR (Deflated Sharpe Ratio) interprets as P(SR > SR*) after selection bias.")
    print(f"Assumed n_trials={n_trials}, target_sharpe={target_sharpe}")
    print("-" * 60)
    pt = out['per_trade']
    print(f"Per-trade: DSR={pt.get('dsr')}, SR={pt.get('sharpe')}, SR*={pt.get('sr_star')}, n={pt.get('n_obs')}")
    pd_ = out['per_day']
    print(f"Per-day:   DSR={pd_.get('dsr')}, SR={pd_.get('sharpe')}, SR*={pd_.get('sr_star')}, n={pd_.get('n_obs')}")

    if pt.get('dsr') is not None and pt['dsr'] < 0.5:
        print("\n⚠️  DSR < 0.5: weak evidence your Sharpe is real after selection bias")
    return out

def run_cmd(cmd, check=True):
    """Run CLI command"""
    cmd = list(cmd)
    if cmd and cmd[0] == "python":
        cmd[0] = sys.executable
    elif cmd and cmd[0] == "pytest":
        cmd = [sys.executable, "-m", "pytest"] + cmd[1:]

    env = os.environ.copy()
    repo_root = str(REPO_ROOT)
    existing = env.get("PYTHONPATH", "")
    if repo_root not in existing.split(os.pathsep):
        env["PYTHONPATH"] = repo_root + (os.pathsep + existing if existing else "")
    env.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".mplconfig"))

    print(">> ", " ".join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True, cwd=repo_root, env=env)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}")
    return p

def read_json(path):
    with open(path, "r") as f:
        return json.load(f)

def read_parquet(path, columns=None):
    return pd.read_parquet(path, columns=columns)

def print_section(title):
    print("\n" + "="*80)
    print(title.center(80))
    print("="*80)

def print_stats_table(data, title=""):
    """Print formatted statistics table"""
    if title:
        print(f"\n{title}:")
    print("-" * 60)
    for key, value in data.items():
        if isinstance(value, float):
            print(f"  {key:.<50} {value:.4f}")
        elif isinstance(value, int):
            print(f"  {key:.<50} {value:,}")
        else:
            print(f"  {key:.<50} {value}")
    print("-" * 60)

# Load helper functions
print("Helper functions loaded ✓")

Helper functions loaded ✓


## 2) Statistics Analysis Functions

In [3]:
def analyze_data_quality(bar_size="1m"):
    """Print comprehensive data quality statistics"""
    print_section(f"DATA QUALITY ANALYSIS ({bar_size})")
    
    bs_dir = RUN_DIR / f"bar_size={bar_size}"
    
    # Load data
    bars = read_parquet(bs_dir / "bars.parquet")
    qa = read_json(bs_dir / "qa_report.json")
    
    # Basic stats
    stats_data = {
        "Total bars": len(bars),
        "Date range start": str(bars.index.min()),
        "Date range end": str(bars.index.max()),
        "Duration (days)": (bars.index.max() - bars.index.min()).days,
        "Synthetic bars": int(bars['is_synthetic'].sum()),
        "Synthetic %": 100 * bars['is_synthetic'].mean(),
        "NaN in OHLCV": bars[['open','high','low','close','volume']].isna().sum().sum(),
    }
    print_stats_table(stats_data, "Basic Statistics")
    
    # Daily coverage
    bars_chi = bars.copy()
    bars_chi.index = bars_chi.index.tz_convert("America/Chicago")
    daily_counts = bars_chi.groupby(bars_chi.index.date).size()
    
    daily_stats = {
        "Trading days": len(daily_counts),
        "Bars/day min": daily_counts.min(),
        "Bars/day median": daily_counts.median(),
        "Bars/day max": daily_counts.max(),
        "Bars/day std": daily_counts.std(),
    }
    print_stats_table(daily_stats, "Daily Coverage")
    
    # QA checks
    print("\nQA Checks:")
    if 'checks' in qa:
        for check_name, result in qa['checks'].items():
            status = result.get('status', '?')
            print(f"  {check_name:.<50} {status}")
    
    return bars

def analyze_features(bar_size="1m"):
    """Print feature statistics"""
    print_section(f"FEATURE ANALYSIS ({bar_size})")
    
    bs_dir = RUN_DIR / f"bar_size={bar_size}"
    feats = read_parquet(bs_dir / "features.parquet")
    
    schema_path = bs_dir / "feature_schema.json"
    if schema_path.exists():
        schema = read_json(schema_path)
        cur_hash = config_hash(FEATURES_YAML)
        schema_hash = schema.get("config_hash")
        if schema_hash and schema_hash != cur_hash:
            print("\n⚠️  STALE FEATURES ARTIFACTS")
            print(f"   feature_schema.json.config_hash={schema_hash}")
            print(f"   current features.yaml hash        ={cur_hash}")
            print("   → Rerun build-features to align artifacts with config")
    
    feat_stats = {
        "Total rows": len(feats),
        "Total features": len(feats.columns),
        "Usable rows": int(feats['usable_for_training'].sum()),
        "Usable %": 100 * feats['usable_for_training'].mean(),
    }
    print_stats_table(feat_stats, "Feature Statistics")
    
    # NaN analysis
    feat_cols = [c for c in feats.columns if c not in ['is_synthetic', 'usable_for_training']]
    nan_counts = feats[feat_cols].isna().sum().sort_values(ascending=False)
    
    print("\nTop 10 Features by NaN Count:")
    print("-" * 60)
    for col, count in nan_counts.head(10).items():
        pct = 100 * count / len(feats)
        print(f"  {col:.<40} {count:>6,} ({pct:>5.2f}%)")
    
    return feats

def analyze_labels(bar_size="1m"):
    """Print label and weight statistics"""
    print_section(f"LABEL & WEIGHT ANALYSIS ({bar_size})")
    
    bs_dir = RUN_DIR / f"bar_size={bar_size}"
    
    label_schema_path = bs_dir / "label_schema.json"
    if label_schema_path.exists():
        label_schema = read_json(label_schema_path)
        label_cfg = label_schema.get("config_snapshot", {})
        primary_cfg = label_cfg.get("primary_labeling", {})
        tb_cfg = primary_cfg.get("triple_barrier", {})
        print("\nLabel config snapshot (from label_schema.json):")
        print("-" * 60)
        print(f"  event_policy............. {primary_cfg.get('event_policy')}")
        if primary_cfg.get("event_policy") == "cusum":
            print(f"  cusum.threshold_atr_mult. {primary_cfg.get('cusum', {}).get('threshold_atr_mult')}")
        if primary_cfg.get("event_policy") == "trend_scanning":
            ts_cfg = primary_cfg.get('trend_scanning', {}) or {}
            print(f"  trend.tstat_threshold..... {ts_cfg.get('tstat_threshold')}")
            print(f"  trend.use_cusum_prefilter. {ts_cfg.get('use_cusum_prefilter')}")
            print(f"  trend.cusum_atr_mult...... {ts_cfg.get('cusum_threshold_atr_mult')}")
            print("\nLOOKAHEAD WARNING:")
            print("  trend_scanning sets `side` using forward returns (future info).")
            print("  Backtests that trade using events['side'] will be biased.")
            print("  Use event_policy='cusum' for unbiased backtests, or predict `side` from features.")
        print(f"  pt_multipliers........... {tb_cfg.get('pt_multipliers')}")
        print(f"  sl_multipliers........... {tb_cfg.get('sl_multipliers')}")
        print(f"  horizon_bars[{bar_size}]......... {tb_cfg.get('horizon_bars', {}).get(bar_size)}")
        
        cur_hash = config_hash(LABELING_YAML)
        schema_hash = hash_content(label_cfg)
        if schema_hash != cur_hash:
            print("\n⚠️  STALE LABELS ARTIFACTS")
            print(f"   label_schema.json.config_snapshot hash={schema_hash}")
            print(f"   current labeling.yaml hash           ={cur_hash}")
            print("   → Rerun build-labels (and build-weights) to align artifacts with config")
    events = read_parquet(bs_dir / "events.parquet")
    weights = read_parquet(bs_dir / "weights.parquet")
    
    # Label distribution
    print("\nLabel Distribution:")
    print("-" * 60)
    label_counts = events['y'].value_counts().sort_index()
    for label, count in label_counts.items():
        pct = 100 * count / len(events)
        label_str = {-1.0: "STOP-FIRST", 0.0: "VERTICAL", 1.0: "TARGET-FIRST"}.get(label, str(label))
        print(f"  y={label:>4} ({label_str:.<30}) {count:>8,} ({pct:>5.1f}%)")
    
    if 'side' in events.columns:
        print("\nSide Distribution:")
        print("-" * 60)
        side_counts = events['side'].value_counts().sort_index()
        for side, count in side_counts.items():
            pct = 100 * count / len(events)
            side_str = {-1: 'SHORT', 1: 'LONG'}.get(int(side), str(side))
            print(f"  side={int(side):>2} ({side_str:.<30}) {count:>8,} ({pct:>5.1f}%)")
    
    # Check if label bias exists
    stop_pct = label_counts.get(-1.0, 0) / len(events) * 100
    target_pct = label_counts.get(1.0, 0) / len(events) * 100
    if stop_pct > 55:
        print(f"\n⚠️  WARNING: Stop-first dominates ({stop_pct:.1f}% vs {target_pct:.1f}%)")
        print("   → Consider widening PT or tightening SL multipliers")
    
    # Weight statistics
    weight_stats = {
        "Weight min": weights['w_final'].min(),
        "Weight 25%": weights['w_final'].quantile(0.25),
        "Weight median": weights['w_final'].median(),
        "Weight 75%": weights['w_final'].quantile(0.75),
        "Weight max": weights['w_final'].max(),
        "Weight std": weights['w_final'].std(),
        "Weight mean": weights['w_final'].mean(),
    }
    print_stats_table(weight_stats, "Weight Statistics (w_final)")
    
    # Check if weights are nearly uniform
    if weights['w_final'].std() / weights['w_final'].mean() < 0.1:
        print("\nℹ️  NOTE: Weights are nearly uniform (low variance)")
        print("   → Uniqueness weighting has minimal effect")
    
    return events, weights

def analyze_training(bar_size="1m", cv_kind="purged_kfold"):
    """Print training statistics"""
    print_section(f"TRAINING ANALYSIS ({bar_size}, {cv_kind})")
    
    bs_dir = RUN_DIR / f"bar_size={bar_size}"
    summary = read_json(bs_dir / "training" / cv_kind / "summary.json")
    
    training_schema_path = bs_dir / "training" / cv_kind / "training_schema.json"
    if training_schema_path.exists():
        training_schema = read_json(training_schema_path)
        cur_hash = config_hash(TRAINING_YAML)
        schema_hash = training_schema.get("config_hash")
        if schema_hash and schema_hash != cur_hash:
            print("\n⚠️  STALE TRAINING ARTIFACTS")
            print(f"   training_schema.json.config_hash={schema_hash}")
            print(f"   current training.yaml hash        ={cur_hash}")
            print("   → Rerun build-train to align artifacts with config")
    
    # Primary model
    print("\nPRIMARY MODEL - Mean Metrics:")
    print("-" * 60)
    for metric, value in summary['metrics_mean'].items():
        status = ""
        if value is None:
            print(f"  {metric:.<40} None")
            continue
        if metric in ['roc_auc', 'roc_auc_target_vs_rest']:
            if value < 0.52:
                status = " ❌ (near-random)"
            elif value < 0.55:
                status = " ⚠️  (weak)"
            elif value < 0.60:
                status = " ⚡ (moderate)"
            else:
                status = " ✅ (good)"
        elif metric == 'f1_macro' and value < 0.20:
            status = " ⚠️  (low)"
        print(f"  {metric:.<40} {value:.4f}{status}")
    
    # Per-split breakdown
    print("\nPer-Split Summary:")
    print("-" * 60)
    for split in summary['metrics_by_split']:
        sid = split['split_id']
        m = split.get('metrics', {})
        auc = m.get('roc_auc', m.get('roc_auc_target_vs_rest'))
        bal_acc = m.get('balanced_accuracy')
        f1m = m.get('f1_macro')
        auc_str = f"{auc:.4f}" if isinstance(auc, (int, float)) else "N/A"
        ba_str = f"{bal_acc:.4f}" if isinstance(bal_acc, (int, float)) else "N/A"
        f1_str = f"{f1m:.4f}" if isinstance(f1m, (int, float)) else "N/A"
        print(f"  Split {sid}: AUC={auc_str}, BAcc={ba_str}, F1m={f1_str}")
    
    # Meta model
    if summary.get('meta', {}).get('enabled'):
        print("\nMETA MODEL - Mean Metrics:")
        print("-" * 60)
        for metric, value in summary['meta']['metrics_mean'].items():
            if isinstance(value, (int, float)):
                status = ""
                if metric == 'roc_auc' and value is not None and value < 0.50:
                    status = " ❌ (anti-predictive)"
                print(f"  {metric:.<40} {value:.4f}{status}")
        
        print("\nMeta Coverage per Split:")
        print("-" * 60)
        for split in summary['meta']['metrics_by_split']:
            sid = split['split_id']
            proposed = split['n_proposed_trades_test']
            accepted = split['n_meta_positive_test']
            rate = split.get('acceptance_rate', 0)
            print(f"  Split {sid}: {accepted:>5,}/{proposed:>6,} accepted ({rate:>5.1%})")
    
    return summary

def analyze_backtest(bar_size="1m", cv_kind="purged_kfold"):
    """Print backtest statistics"""
    print_section(f"BACKTEST ANALYSIS ({bar_size}, {cv_kind})")
    
    bs_dir = RUN_DIR / f"bar_size={bar_size}"
    summary = read_json(bs_dir / "backtests" / cv_kind / "summary.json")
    
    backtest_schema_path = bs_dir / "backtests" / cv_kind / "backtest_schema.json"
    if backtest_schema_path.exists():
        backtest_schema = read_json(backtest_schema_path)
        cur_hash = config_hash(BACKTEST_YAML)
        schema_hash = backtest_schema.get("backtest_config_hash")
        if schema_hash and schema_hash != cur_hash:
            print("\n⚠️  STALE BACKTEST ARTIFACTS")
            print(f"   backtest_schema.json.backtest_config_hash={schema_hash}")
            print(f"   current backtest.yaml hash              ={cur_hash}")
            print("   → Rerun build-backtest to align artifacts with config")
        decision = backtest_schema.get("backtest_config", {}).get("decision", {})
        print("\nBacktest decision snapshot (from backtest_schema.json):")
        print("-" * 60)
        print(f"  use_meta................. {decision.get('use_meta')}")
        print(f"  primary_score_column...... {decision.get('primary_score_column')}")
        print(f"  primary_threshold......... {decision.get('primary_threshold')}")
        print(f"  meta_threshold............ {decision.get('meta_threshold')}")
        print(f"  require_meta_for_trade.... {decision.get('require_meta_for_trade')}")
    
    # Aggregate stats
    total_trades = sum(s['trades_count'] for s in summary['metrics_by_split'])
    total_pnl = sum(s.get('total_pnl_usd', 0) for s in summary['metrics_by_split'])
    
    agg_stats = {
        "Total trades": total_trades,
        "Total PnL USD": total_pnl,
        "Avg PnL/trade": total_pnl / total_trades if total_trades > 0 else 0,
    }
    print_stats_table(agg_stats, "Aggregate Backtest Results")
    
    # Per-split breakdown
    print("\nPer-Split Results:")
    print("-" * 80)
    print(f"{'Split':<8} {'Trades':<10} {'PnL (USD)':<15} {'Win Rate':<12} {'Profit Factor':<15}")
    print("-" * 80)
    
    for split in summary['metrics_by_split']:
        sid = split['split_id']
        trades = split['trades_count']
        pnl = split.get('total_pnl_usd', 0)
        wr = split.get('win_rate')
        pf = split.get('profit_factor')
        
        wr_str = f"{wr:.1%}" if wr is not None else "N/A"
        pf_str = f"{pf:.2f}" if pf is not None else "N/A"
        
        print(f"{sid:<8} {trades:<10,} ${pnl:<14,.0f} {wr_str:<12} {pf_str:<15}")
    
    # Load actual trades for detailed analysis
    all_trades = []
    backtest_base = bs_dir / "backtests" / cv_kind
    for trades_path in sorted(backtest_base.glob("*/trades.parquet")):
        df = read_parquet(trades_path)
        all_trades.append(df)
    
    if all_trades:
        trades_df = pd.concat(all_trades, ignore_index=True)
        
        # Calculate holding_bars if not present (for executed trades only)
        if 'holding_bars' not in trades_df.columns and 'entry_ts' in trades_df.columns and 'exit_ts' in trades_df.columns:
            # Extract bar size from path to determine time interval
            bar_minutes = {'1m': 1, '5m': 5, '15m': 15, '1h': 60}.get(bar_size, 5)
            executed_mask = trades_df['executed'] == True
            trades_df.loc[executed_mask, 'holding_bars'] = (
                (trades_df.loc[executed_mask, 'exit_ts'] - trades_df.loc[executed_mask, 'entry_ts']).dt.total_seconds() / 60 / bar_minutes
            )
        
        trade_stats = {
            "PnL min": trades_df['pnl_usd'].min(),
            "PnL 25%": trades_df['pnl_usd'].quantile(0.25),
            "PnL median": trades_df['pnl_usd'].median(),
            "PnL 75%": trades_df['pnl_usd'].quantile(0.75),
            "PnL max": trades_df['pnl_usd'].max(),
        }
        
        # Add holding bars stats if available
        if 'holding_bars' in trades_df.columns:
            executed_df = trades_df[trades_df['executed'] == True]
            if len(executed_df) > 0 and executed_df['holding_bars'].notna().any():
                trade_stats["Holding bars median"] = executed_df['holding_bars'].median()
                trade_stats["Holding bars max"] = executed_df['holding_bars'].max()
        
        print_stats_table(trade_stats, "Trade-Level Statistics")
        
        # Exit sources
        if 'exit_source' in trades_df.columns:
            print("\nExit Sources:")
            print("-" * 60)
            for source, count in trades_df['exit_source'].value_counts().items():
                pct = 100 * count / len(trades_df)
                print(f"  {source:.<40} {count:>5,} ({pct:>5.1f}%)")
    
    # Warning if losing money
    if total_pnl < 0:
        print(f"\n❌ WARNING: Strategy is LOSING MONEY (${total_pnl:,.0f})")
        print("   → Do NOT deploy to live trading")
    
    return summary, trades_df if all_trades else None

def analyze_walk_forward(bar_size="1m"):
    """Print walk-forward statistics (robust to skipped-only summaries)."""
    print_section(f"WALK-FORWARD ANALYSIS ({bar_size})")
    
    wf_dir = RUN_DIR / "walkforward" / f"bar_size={bar_size}"
    summary_path = wf_dir / "summary.json"
    if not summary_path.exists():
        print("Missing:", summary_path)
        return None
    summary = read_json(summary_path)

    windows = summary.get('metrics_by_window', []) or []
    n_windows = int(summary.get('n_windows', len(windows)))
    metrics_mean = summary.get('metrics_mean', {}) or {}

    statuses = [w.get('status') for w in windows if isinstance(w, dict) and w.get('status')]
    status_counts = pd.Series(statuses).value_counts().to_dict() if statuses else {}
    skipped_reasons = [w.get('reason') for w in windows if w.get('status') == 'skipped' and w.get('reason')]
    reason_counts = pd.Series(skipped_reasons).value_counts().to_dict() if skipped_reasons else {}

    trades_counts = [w.get('trades_count') for w in windows if w.get('trades_count') is not None]
    pnl_vals = [w.get('total_pnl_usd') for w in windows if w.get('total_pnl_usd') is not None]
    skipped_counts = [w.get('skipped_count') for w in windows if w.get('skipped_count') is not None]

    wf_stats = {
        "Total windows": n_windows,
        "Windows ok": int(status_counts.get('ok', 0)),
        "Windows skipped": int(status_counts.get('skipped', 0)),
        "Mean n_train": float(metrics_mean.get('n_train', np.nan)),
        "Mean n_test": float(metrics_mean.get('n_test', np.nan)),
    }
    if trades_counts:
        wf_stats["Mean trades/window"] = float(np.nanmean(pd.to_numeric(trades_counts, errors='coerce')))
    if skipped_counts:
        wf_stats["Mean skipped/window"] = float(np.nanmean(pd.to_numeric(skipped_counts, errors='coerce')))
    if pnl_vals:
        wf_stats["Mean PnL/window"] = float(np.nanmean(pd.to_numeric(pnl_vals, errors='coerce')))
    print_stats_table(wf_stats, "Walk-Forward Statistics")

    if reason_counts:
        print("\nTop skip reasons:")
        for k, v in list(reason_counts.items())[:5]:
            print(f"  {k:.<40} {v}")

    active = [w for w in windows if (w.get('trades_count') or 0) > 0]
    pct_active = 100.0 * len(active) / n_windows if n_windows else 0.0
    print(f"\nWindows with trades: {len(active)} / {n_windows} ({pct_active:.1f}%)")

    if active:
        pnls = pd.Series(pd.to_numeric([w.get('total_pnl_usd') for w in active], errors='coerce')).dropna().tolist()
        if pnls:
            winning_windows = sum(1 for p in pnls if p > 0)
            active_stats = {
                "PnL min": min(pnls),
                "PnL median": float(np.median(pnls)),
                "PnL max": max(pnls),
                "Total PnL": sum(pnls),
                "Winning windows": winning_windows,
                "Win rate": 100 * winning_windows / len(pnls),
            }
            print_stats_table(active_stats, "Active Window Statistics")
    else:
        # Helpful hint when everything is skipped
        try:
            wf_cfg = read_config(WALKFORWARD_YAML)
            min_train_events = (wf_cfg.get('schedule', {}) or {}).get('min_train_events')
            if min_train_events is not None:
                print(f"\nNo windows produced trades. walkforward.yaml schedule.min_train_events={min_train_events}")
        except Exception:
            pass

    if pct_active < 30 and n_windows:
        print(f"\n⚠️  WARNING: Only {pct_active:.1f}% of windows produce trades")
        print("   → Strategy fires too rarely for live deployment")
    
    return summary

print("Statistics analysis functions loaded ✓")

Statistics analysis functions loaded ✓


## 3) Monte Carlo Simulation Functions

In [4]:
def monte_carlo_trade_sequence(trades_df, n_simulations=1000, seed=42, block_size=None):
    """
    Monte Carlo simulation using BLOCK BOOTSTRAP for time series data.
    
    CRITICAL: For time series, we must preserve temporal dependencies!
    - Creates overlapping blocks of consecutive trades
    - Samples blocks with replacement (not individual trades)
    - Preserves ordering within each block
    
    Args:
        trades_df: DataFrame with executed trades
        n_simulations: Number of bootstrap iterations
        seed: Random seed
        block_size: Size of blocks (default: sqrt(n_trades), bounded 5-20)
    """
    print_section("MONTE CARLO: BLOCK BOOTSTRAP (Time Series)")
    
    if trades_df is None or len(trades_df) == 0:
        print("No trades available for Monte Carlo simulation")
        return None
    
    # Filter to executed trades only
    executed_df = trades_df[trades_df['executed'] == True].copy()
    if len(executed_df) == 0:
        print("No executed trades available for Monte Carlo simulation")
        return None
    
    np.random.seed(seed)
    
    # Sort by entry time to preserve temporal order
    executed_df = executed_df.sort_values('entry_ts')
    pnls = executed_df['pnl_usd'].dropna().values
    n_trades = len(pnls)
    
    if n_trades == 0:
        print("No valid PnL values for Monte Carlo simulation")
        return None
    
    # Determine block size (preserve temporal dependencies)
    if block_size is None:
        # Default: sqrt(n_trades), bounded between 5 and 20
        block_size = max(5, min(20, int(np.sqrt(n_trades))))
    
    if n_trades < block_size:
        block_size = n_trades
        print(f"⚠️  Only {n_trades} trades - using block_size={block_size}")
    
    # Create overlapping blocks (sliding window to preserve more structure)
    n_blocks = max(1, n_trades - block_size + 1)
    blocks = []
    for i in range(n_blocks):
        blocks.append(pnls[i:i + block_size])
    
    # How many blocks needed to reconstruct a full sequence
    blocks_needed = int(np.ceil(n_trades / block_size))
    
    print(f"Running {n_simulations:,} simulations on {n_trades:,} trades")
    print(f"Block bootstrap: block_size={block_size}, n_blocks={n_blocks}")
    print(f"This preserves temporal dependencies within {block_size}-trade windows\n")
    
    # Storage for simulation results
    sim_total_pnl = []
    sim_sharpe = []
    sim_max_dd = []
    sim_win_rate = []
    
    for i in range(n_simulations):
        # Block bootstrap: sample blocks with replacement
        sampled_blocks = []
        for _ in range(blocks_needed):
            block_idx = np.random.randint(0, len(blocks))
            sampled_blocks.append(blocks[block_idx])
        
        # Concatenate blocks and trim to original size
        shuffled_pnls = np.concatenate(sampled_blocks)[:n_trades]
        
        # Compute metrics
        total_pnl = shuffled_pnls.sum()
        sharpe = np.mean(shuffled_pnls) / (np.std(shuffled_pnls) + 1e-8) * np.sqrt(252)  # Annualized
        
        # Max drawdown
        cumulative = np.cumsum(shuffled_pnls)
        running_max = np.maximum.accumulate(cumulative)
        drawdown = running_max - cumulative
        max_dd = drawdown.max()
        
        win_rate = (shuffled_pnls > 0).mean()
        
        sim_total_pnl.append(total_pnl)
        sim_sharpe.append(sharpe)
        sim_max_dd.append(max_dd)
        sim_win_rate.append(win_rate)
    
    # Convert to arrays
    sim_total_pnl = np.array(sim_total_pnl)
    sim_sharpe = np.array(sim_sharpe)
    sim_max_dd = np.array(sim_max_dd)
    sim_win_rate = np.array(sim_win_rate)
    
    # Calculate confidence intervals
    ci_95_pnl = (np.percentile(sim_total_pnl, 2.5), np.percentile(sim_total_pnl, 97.5))
    ci_95_sharpe = (np.percentile(sim_sharpe, 2.5), np.percentile(sim_sharpe, 97.5))
    ci_95_dd = (np.percentile(sim_max_dd, 2.5), np.percentile(sim_max_dd, 97.5))
    
    # Print results
    mc_stats = {
        "Actual Total PnL": pnls.sum(),
        "MC Median PnL": np.median(sim_total_pnl),
        "MC Mean PnL": np.mean(sim_total_pnl),
        "MC PnL 95% CI Lower": ci_95_pnl[0],
        "MC PnL 95% CI Upper": ci_95_pnl[1],
        "MC Sharpe Median": np.median(sim_sharpe),
        "MC Sharpe 95% CI Lower": ci_95_sharpe[0],
        "MC Sharpe 95% CI Upper": ci_95_sharpe[1],
        "MC Max DD Median": np.median(sim_max_dd),
        "MC Max DD 95% CI Lower": ci_95_dd[0],
        "MC Max DD 95% CI Upper": ci_95_dd[1],
        "Probability of Profit": (sim_total_pnl > 0).mean() * 100,
    }
    print_stats_table(mc_stats, "Monte Carlo Results")
    
    # Plot distributions
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # PnL distribution
    axes[0, 0].hist(sim_total_pnl, bins=50, alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(pnls.sum(), color='red', linestyle='--', label='Actual', linewidth=2)
    axes[0, 0].axvline(ci_95_pnl[0], color='orange', linestyle=':', label='95% CI')
    axes[0, 0].axvline(ci_95_pnl[1], color='orange', linestyle=':')
    axes[0, 0].set_xlabel('Total PnL (USD)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title(f'Distribution of Total PnL (Block Bootstrap, size={block_size})')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Sharpe distribution
    axes[0, 1].hist(sim_sharpe, bins=50, alpha=0.7, edgecolor='black')
    axes[0, 1].axvline(ci_95_sharpe[0], color='orange', linestyle=':', label='95% CI')
    axes[0, 1].axvline(ci_95_sharpe[1], color='orange', linestyle=':')
    axes[0, 1].set_xlabel('Sharpe Ratio')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution of Sharpe Ratio')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Max DD distribution
    axes[1, 0].hist(sim_max_dd, bins=50, alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(ci_95_dd[0], color='orange', linestyle=':', label='95% CI')
    axes[1, 0].axvline(ci_95_dd[1], color='orange', linestyle=':')
    axes[1, 0].set_xlabel('Max Drawdown (USD)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Max Drawdown')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Win rate distribution
    axes[1, 1].hist(sim_win_rate * 100, bins=50, alpha=0.7, edgecolor='black')
    axes[1, 1].axvline((pnls > 0).mean() * 100, color='red', linestyle='--', label='Actual', linewidth=2)
    axes[1, 1].set_xlabel('Win Rate (%)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Distribution of Win Rate')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Interpretation
    prob_profit = (sim_total_pnl > 0).mean() * 100
    if prob_profit < 50:
        print(f"\n❌ CRITICAL: Only {prob_profit:.1f}% probability of profit")
        print("   → Strategy is more likely to LOSE money than make money")
    elif prob_profit < 70:
        print(f"\n⚠️  WARNING: {prob_profit:.1f}% probability of profit (moderate risk)")
    else:
        print(f"\n✅ GOOD: {prob_profit:.1f}% probability of profit")
    
    return {
        'total_pnl': sim_total_pnl,
        'sharpe': sim_sharpe,
        'max_dd': sim_max_dd,
        'win_rate': sim_win_rate,
        'block_size': block_size,
        'n_blocks': n_blocks,
    }

def monte_carlo_equity_curve(trades_df, n_simulations=100, seed=42, block_size=None):
    """
    Monte Carlo simulation of equity curves using BLOCK BOOTSTRAP.
    Visualizes the 'cone of uncertainty' while preserving temporal structure.
    """
    print_section("MONTE CARLO: EQUITY CURVE UNCERTAINTY (Block Bootstrap)")
    
    if trades_df is None or len(trades_df) == 0:
        print("No trades available")
        return None
    
    # Filter to executed trades only
    executed_df = trades_df[trades_df['executed'] == True].copy()
    if len(executed_df) == 0:
        print("No executed trades available for Monte Carlo simulation")
        return None
    
    np.random.seed(seed)
    
    # Sort by entry time to preserve temporal order
    executed_df = executed_df.sort_values('entry_ts')
    pnls = executed_df['pnl_usd'].dropna().values
    n_trades = len(pnls)
    
    if n_trades == 0:
        print("No valid PnL values for Monte Carlo simulation")
        return None
    
    # Determine block size
    if block_size is None:
        block_size = max(5, min(20, int(np.sqrt(n_trades))))
    
    if n_trades < block_size:
        block_size = n_trades
    
    # Create overlapping blocks
    n_blocks = max(1, n_trades - block_size + 1)
    blocks = []
    for i in range(n_blocks):
        blocks.append(pnls[i:i + block_size])
    
    blocks_needed = int(np.ceil(n_trades / block_size))
    
    print(f"Simulating {n_simulations} equity curves on {n_trades} trades")
    print(f"Block size: {block_size} (preserves temporal dependencies)\n")
    
    # Storage
    equity_curves = np.zeros((n_simulations, n_trades))
    
    for i in range(n_simulations):
        # Block bootstrap
        sampled_blocks = []
        for _ in range(blocks_needed):
            block_idx = np.random.randint(0, len(blocks))
            sampled_blocks.append(blocks[block_idx])
        
        shuffled_pnls = np.concatenate(sampled_blocks)[:n_trades]
        equity_curves[i, :] = np.cumsum(shuffled_pnls)
    
    # Calculate percentiles
    p05 = np.percentile(equity_curves, 5, axis=0)
    p25 = np.percentile(equity_curves, 25, axis=0)
    p50 = np.percentile(equity_curves, 50, axis=0)
    p75 = np.percentile(equity_curves, 75, axis=0)
    p95 = np.percentile(equity_curves, 95, axis=0)
    
    # Actual equity curve
    actual_equity = np.cumsum(pnls)
    
    # Plot
    plt.figure(figsize=(14, 8))
    
    # Plot some individual simulations (faint)
    for i in range(min(20, n_simulations)):
        plt.plot(equity_curves[i, :], color='gray', alpha=0.1, linewidth=0.5)
    
    # Plot percentiles
    plt.fill_between(range(n_trades), p05, p95, alpha=0.3, color='blue', label='90% CI')
    plt.fill_between(range(n_trades), p25, p75, alpha=0.4, color='blue', label='50% CI')
    plt.plot(p50, color='blue', linewidth=2, label='Median')
    plt.plot(actual_equity, color='red', linewidth=2, linestyle='--', label='Actual')
    
    plt.axhline(0, color='black', linestyle=':', linewidth=1)
    plt.xlabel('Trade Number')
    plt.ylabel('Cumulative PnL (USD)')
    plt.title(f'Monte Carlo Equity Curve: Block Bootstrap (size={block_size})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nℹ️  The 'cone of uncertainty' shows possible equity paths")
    print("   Block bootstrap preserves temporal dependencies within blocks")
    print("   Wider cone = higher variance = less predictable outcomes")

print("Monte Carlo simulation functions loaded ✓")

Monte Carlo simulation functions loaded ✓


## 4) Run Pipeline Stages
Execute each stage and immediately print stats.

### 4.1 Build Data

In [5]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-data",
    "--config", str(DATA_YAML),
    "--run-id", RUN_ID,
    "--seed", str(SEED),
])

# Analyze data quality
bars = analyze_data_quality("1m")

>>  /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-data --config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/data.yaml --run-id run_20251227_042529_c5a31e4b --seed 42
2025-12-26 22:25:30 [INFO] __main__: ================================================================================
2025-12-26 22:25:30 [INFO] __main__: V3 DATA PIPELINE - BUILD DATA
2025-12-26 22:25:30 [INFO] __main__: ================================================================================
2025-12-26 22:25:30 [INFO] __main__: Loaded config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/data.yaml
2025-12-26 22:25:30 [INFO] __main__: Run ID: run_20251227_042529_c5a31e4b
2025-12-26 22:25:30 [INFO] __main__: Output directory: runs/run_20251227_042529_c5a31e4b
2025-12-26 22:25:30 [INFO] __main__: Canonical bar size: 1m
2025-12-26 22:25:30 [INFO] __main__: Bar sizes to process: ['1m']
2025-12

### 4.2 Build Features

In [6]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-features",
    "--run-dir", str(RUN_DIR),
    "--features-config", str(FEATURES_YAML),
])

# Analyze features
feats = analyze_features("1m")

>>  /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-features --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b --features-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/features.yaml
2025-12-26 22:25:55 [INFO] __main__: ================================================================================
2025-12-26 22:25:55 [INFO] __main__: V3 FEATURE PIPELINE - BUILD FEATURES
2025-12-26 22:25:55 [INFO] __main__: ================================================================================
2025-12-26 22:25:55 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b
2025-12-26 22:25:55 [INFO] __main__: Loaded features config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/features.yaml
2025-12-26 22:25:55 [INFO] __main__: Found existing manifest with

### 4.3 Build Labels

In [7]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-labels",
    "--run-dir", str(RUN_DIR),
    "--labeling-config", str(LABELING_YAML),
    "--execution-spec", str(EXECUTION_SPEC_YAML),
])

run_cmd([
    "python", "-m", CLI_MODULE, "build-weights",
    "--run-dir", str(RUN_DIR),
    "--labeling-config", str(LABELING_YAML),
])

# Analyze labels
events, weights = analyze_labels("1m")

>>  /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-labels --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b --labeling-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/labeling.yaml --execution-spec /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/execution_spec.yaml
2025-12-26 22:26:02 [INFO] __main__: ================================================================================
2025-12-26 22:26:02 [INFO] __main__: V3 LABEL PIPELINE - BUILD LABELS
2025-12-26 22:26:02 [INFO] __main__: ================================================================================
2025-12-26 22:26:02 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b
2025-12-26 22:26:02 [INFO] __main__: Loaded labeling config from /Users/eshaanganguly/Documents/projects/algos 3 

### 4.4 Build Weights

In [8]:
# Weights are built and analyzed in the labels section above
print("✓ Weights built (see labels section)")

✓ Weights built (see labels section)


### 4.5 Build CV Splits

In [9]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-cv",
    "--run-dir", str(RUN_DIR),
    "--validation-config", str(VALIDATION_YAML),
])

print("✓ CV splits created (see training analysis for details)")

>>  /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-cv --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b --validation-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/validation.yaml
2025-12-26 22:26:09 [INFO] __main__: ================================================================================
2025-12-26 22:26:09 [INFO] __main__: V3 VALIDATION PIPELINE - BUILD CV
2025-12-26 22:26:09 [INFO] __main__: ================================================================================
2025-12-26 22:26:09 [INFO] __main__: Run directory: /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b
2025-12-26 22:26:09 [INFO] __main__: Loaded validation config from /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/validation.yaml
2025-12-26 22:26:09 [INFO] __main__: Found existing manifest with 

### 4.6 Train Models

In [10]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-train",
    "--run-dir", str(RUN_DIR),
    "--training-config", str(TRAINING_YAML),
    "--cv-kind", CV_KIND,
])

# Analyze training
train_summary = analyze_training("1m", CV_KIND)

>>  /Users/eshaanganguly/Documents/projects/algos 3 topstep/.venv/bin/python -m ml_intraday_v3.cli build-train --run-dir /Users/eshaanganguly/Documents/projects/algos 3 topstep/runs/run_20251227_042529_c5a31e4b --training-config /Users/eshaanganguly/Documents/projects/algos 3 topstep/ml_intraday_v3/configs/training.yaml --cv-kind purged_kfold


KeyboardInterrupt: 

### 4.6.1 Rare Events Corrections - Overview

**Rare Events Corrections** for logistic regression address bias in predicted probabilities when dealing with class imbalance (King & Zeng, 2001).

#### What are Rare Events Corrections?

When training logistic regression on imbalanced data:
- Standard models produce **biased probability estimates**
- Probabilities are systematically miscalibrated
- King & Zeng corrections fix both:
  1. **Sample weighting** during training
  2. **Prior correction** to predicted probabilities

#### When to Use

✅ **Use if:**
- Positive class < 20% (rare events)
- Probabilities matter (not just ranking)
- Known or estimable population prior

❌ **Skip if:**
- Balanced data (40-60% positive)
- Only care about AUC/ranking
- Using non-probabilistic models

#### Configuration

Add to `configs/training.yaml`:

```yaml
model:
  type: "relogit"
  relogit_params:
    tau: null  # Auto-estimate or set to known prior (e.g., 0.05)
    use_sample_weights: true
    weight_method: "king_zeng"
```

See `training/RARE_EVENTS_QUICKSTART.md` for full documentation.

In [ ]:
# Three Ways to Use Rare Events Corrections
# (These are examples - not executed in the pipeline)

from ml_intraday_v3.training.rare_events import (
    RelogitClassifier,
    correct_rare_events_probabilities,
    compute_rare_event_weights
)
from sklearn.linear_model import LogisticRegression

print("\n" + "="*80)
print("RARE EVENTS CORRECTIONS - USAGE EXAMPLES")
print("="*80)

print("\n1. RelogitClassifier (Easiest - Drop-in Replacement)")
print("-" * 60)
print("""
clf = RelogitClassifier(
    tau=0.05,              # 5% historical win rate (or None to auto-estimate)
    use_sample_weights=True,
    random_state=42
)
clf.fit(X_train, y_train)
prob = clf.predict_proba(X_test)[:, 1]  # Automatically corrected!
""")

print("\n2. Post-hoc Correction (Apply After Training)")
print("-" * 60)
print("""
# Train normally
lr = LogisticRegression(random_state=42)
lr.fit(X_train, y_train)
prob_raw = lr.predict_proba(X_test)[:, 1]

# Correct probabilities
prob_corrected = correct_rare_events_probabilities(
    prob_raw,
    y_test,
    tau=0.05  # True prior
)
""")

print("\n3. Sample Weighting Only (During Training)")
print("-" * 60)
print("""
# Compute weights
weights = compute_rare_event_weights(y_train, method="king_zeng", tau=0.05)

# Train with weights
lr = LogisticRegression(random_state=42)
lr.fit(X_train, y_train, sample_weight=weights)
""")

print("\n" + "="*80)
print("NOTE: The pipeline automatically applies corrections if configured.")
print("      Check section 4.6.3 below to see calibration analysis.")
print("="*80)

### 4.6.2 Model Performance Analysis - Cost Curves

Visualize classifier performance across all class distributions and misclassification costs using **cost curves** (Drummond & Holte, 2006).

- **Cost Curve**: Shows Normalized Expected Cost (NC) vs Probability Cost (PC)
- **Lower is better**: AUCC (Area Under Cost Curve) summarizes performance
- **Trading-specific**: Maps risk/reward ratios to cost ratios

In [ ]:
# Cost Curves Analysis
from ml_intraday_v3.analysis.cost_curves import (
    compute_cost_curve,
    bootstrap_cost_curve,
    plot_cost_curve,
    compute_trading_cost_curve,
    compare_models_cost_curves,
    compute_area_under_cost_curve
)
import pandas as pd
import numpy as np
from pathlib import Path

print("\n" + "="*80)
print("COST CURVE ANALYSIS")
print("="*80)

# Load model predictions from disk
try:
    # The pipeline saves predictions per fold in:
    # RUN_DIR / "bar_size=1m" / "training" / "purged_kfold" / "fold_X" / "preds.parquet"
    
    bar_size_dir = BAR_SIZE_1M_DIR  # Note: uses "bar_size=" prefix
    training_dir = RUN_DIR / bar_size_dir / "training" / CV_KIND  # CV_KIND is "purged_kfold" or "cpcv"
    
    if not training_dir.exists():
        raise FileNotFoundError(
            f"Training directory not found: {training_dir}\n"
            f"Make sure section 4.6 (Train Models) has completed successfully."
        )
    
    # Load predictions from all folds (all are out-of-sample by CV design)
    fold_dirs = sorted([d for d in training_dir.iterdir() if d.is_dir()])
    
    if not fold_dirs:
        raise FileNotFoundError(f"No fold directories found in {training_dir}")
    
    print(f"\nLoading predictions from {len(fold_dirs)} folds...")
    all_preds = []
    
    for fold_dir in fold_dirs:
        preds_file = fold_dir / "preds.parquet"
        if preds_file.exists():
            fold_preds = pd.read_parquet(preds_file)
            all_preds.append(fold_preds)
            print(f"  {fold_dir.name}: {len(fold_preds)} predictions")
    
    if not all_preds:
        raise FileNotFoundError(f"No preds.parquet files found in {training_dir}")
    
    # Combine all folds
    preds_df = pd.concat(all_preds, ignore_index=True)
    print(f"\nTotal predictions: {len(preds_df)}")
    print(f"Columns: {list(preds_df.columns)}")
    
    # Extract labels and probabilities for cost curves
    # y_true: -1 (stop), 0 (vertical), 1 (target)
    # For binary classification: 1 = profitable (hit target), 0 = not profitable
    y_test = (preds_df['y_true'] == 1).astype(int).values
    
    # Use p_target as probability of profitable outcome
    y_pred_proba = preds_df['p_target'].values
    
    print(f"\nLabel distribution:")
    print(f"  Profitable (hit target): {y_test.sum()} ({y_test.mean()*100:.1f}%)")
    print(f"  Not profitable: {(1-y_test).sum()} ({(1-y_test.mean())*100:.1f}%)")
    print(f"\nProbability range: [{y_pred_proba.min():.3f}, {y_pred_proba.max():.3f}]")
    
    # 1. Basic Cost Curve
    print("\n1. Computing cost curve...")
    curve_df = compute_cost_curve(y_test, y_pred_proba)
    aucc = compute_area_under_cost_curve(curve_df)
    
    print(f"   AUCC (Area Under Cost Curve): {aucc:.4f} (lower is better)")
    print(f"   NC range: [{curve_df['nc'].min():.4f}, {curve_df['nc'].max():.4f}]")
    
    # 2. Plot Cost Curve with Bootstrap CI
    print("\n2. Computing bootstrap confidence intervals (this may take 10-30 seconds)...")
    mean_curve, lower_ci, upper_ci = bootstrap_cost_curve(
        y_test, y_pred_proba,
        n_bootstrap=500,  # Use 500 for speed (increase to 1000 for publication)
        confidence_level=0.95,
        random_state=42
    )
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot with confidence intervals
    plot_cost_curve(
        mean_curve,
        ax=ax1,
        label="Primary Model",
        show_confidence=True,
        confidence_bounds=(lower_ci, upper_ci),
        color='blue'
    )
    ax1.set_title('Cost Curve with 95% Bootstrap CI', fontsize=13, fontweight='bold')
    
    # 3. Trading-Specific Cost Curve
    print("\n3. Computing trading-specific cost curve...")
    trading_curve = compute_trading_cost_curve(
        y_test, y_pred_proba,
        risk_reward_ratios=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
    )
    
    # Find optimal RR
    optimal_idx = trading_curve['nc'].idxmin()
    optimal_row = trading_curve.loc[optimal_idx]
    
    print(f"\n   Trading Cost Analysis:")
    print(f"   Optimal Risk/Reward Ratio: {optimal_row['risk_reward_ratio']:.2f}")
    print(f"   At optimal: NC={optimal_row['nc']:.4f}, TPR={optimal_row['tpr']:.3f}, FPR={optimal_row['fpr']:.3f}")
    print(f"   Optimal threshold: {optimal_row['threshold']:.3f}")
    
    # Plot trading cost curve
    plot_cost_curve(trading_curve, ax=ax2, label="Trading Strategy", color='green')
    
    # Add markers for specific RR ratios
    for rr in [1.0, 2.0, 3.0]:
        row = trading_curve[trading_curve['risk_reward_ratio'] == rr].iloc[0]
        ax2.plot(row['pc'], row['nc'], 'ro', markersize=8)
        ax2.annotate(f'RR={rr}', (row['pc'], row['nc']), 
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax2.set_title('Trading Cost Curve (Risk/Reward Ratios)', fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # 4. Summary Table
    print("\n4. Cost Curve Summary by Risk/Reward Ratio:")
    summary = trading_curve[['risk_reward_ratio', 'cost_ratio', 'nc', 'threshold', 'tpr', 'fpr']].copy()
    summary.columns = ['RR', 'Cost Ratio', 'NC', 'Threshold', 'TPR', 'FPR']
    print(summary.to_string(index=False))
    
except FileNotFoundError as e:
    print(f"\n⚠️  {e}")
    print("\nExpected structure:")
    print(f"  RUN_DIR/bar_size=1m/training/{CV_KIND if 'CV_KIND' in dir() else 'purged_kfold'}/fold_X/preds.parquet")
    
except Exception as e:
    print(f"\n⚠️  Error: {e}")
    print(f"\nDebug info:")
    print(f"  RUN_DIR: {RUN_DIR if 'RUN_DIR' in dir() else 'NOT SET'}")
    print(f"  CV_KIND: {CV_KIND if 'CV_KIND' in dir() else 'NOT SET'}")
    import traceback
    traceback.print_exc()


### 4.6.3 Rare Events Calibration Analysis

Analyze the impact of rare events corrections on probability calibration.

This section:
- Compares uncorrected vs corrected probabilities (if available)
- Shows calibration curves
- Computes Brier score improvement
- Validates that corrections improve calibration without hurting discrimination (AUC)

In [ ]:
# Rare Events Calibration Analysis
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.calibration import calibration_curve
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("\n" + "="*80)
print("RARE EVENTS CALIBRATION ANALYSIS")
print("="*80)

try:
    # Load predictions from disk (same as cost curves section)
    bar_size_dir = BAR_SIZE_1M_DIR
    training_dir = RUN_DIR / bar_size_dir / "training" / CV_KIND

    if not training_dir.exists():
        raise FileNotFoundError(
            f"Training directory not found: {training_dir}\n"
            f"Make sure section 4.6 (Train Models) has completed successfully."
        )

    # Load predictions from all folds
    fold_dirs = sorted([d for d in training_dir.iterdir() if d.is_dir()])

    if not fold_dirs:
        raise FileNotFoundError(f"No fold directories found in {training_dir}")

    print(f"\nLoading predictions from {len(fold_dirs)} folds...")
    all_preds = []

    for fold_dir in fold_dirs:
        preds_file = fold_dir / "preds.parquet"
        if preds_file.exists():
            fold_preds = pd.read_parquet(preds_file)
            all_preds.append(fold_preds)

    if not all_preds:
        raise FileNotFoundError(f"No preds.parquet files found in {training_dir}")

    # Combine all folds
    preds_df = pd.concat(all_preds, ignore_index=True)
    print(f"Total predictions: {len(preds_df)}")
    print(f"Columns: {list(preds_df.columns)}")

    # Check if rare events corrections were applied
    has_uncorrected = 'p_target_uncorrected' in preds_df.columns

    if not has_uncorrected:
        print("\n⚠️  NOTE: Rare events corrections were not applied in training.")
        print("    This section requires 'p_target_uncorrected' column.")
        print("    To enable corrections, set model.type='relogit' in training.yaml")
        print("\n    Showing calibration analysis for standard predictions...\n")

    # Extract labels and probabilities
    y_true = (preds_df['y_true'] == 1).astype(int).values
    p_corrected = preds_df['p_target'].values

    # Get uncorrected probabilities if available
    if has_uncorrected:
        p_uncorrected = preds_df['p_target_uncorrected'].values
    else:
        p_uncorrected = p_corrected  # Use same for comparison

    print(f"\nLabel distribution:")
    print(f"  Positive class (hit target): {y_true.sum()} ({y_true.mean()*100:.1f}%)")
    print(f"  Negative class: {(1-y_true).sum()} ({(1-y_true).mean()*100:.1f}%)")

    # Compute metrics
    print(f"\n" + "="*60)
    print("CALIBRATION METRICS")
    print("="*60)

    if has_uncorrected:
        brier_uncorrected = brier_score_loss(y_true, p_uncorrected)
        auc_uncorrected = roc_auc_score(y_true, p_uncorrected)

        brier_corrected = brier_score_loss(y_true, p_corrected)
        auc_corrected = roc_auc_score(y_true, p_corrected)

        print(f"\nUncorrected Probabilities:")
        print(f"  Brier Score: {brier_uncorrected:.4f}")
        print(f"  AUC:         {auc_uncorrected:.4f}")

        print(f"\nCorrected Probabilities (King & Zeng):")
        print(f"  Brier Score: {brier_corrected:.4f}")
        print(f"  AUC:         {auc_corrected:.4f}")

        print(f"\nImprovement:")
        print(f"  Brier Score: {brier_uncorrected - brier_corrected:+.4f} (lower is better)")
        print(f"  AUC:         {auc_corrected - auc_uncorrected:+.4f} (should be ~0)")

        if abs(auc_corrected - auc_uncorrected) > 0.01:
            print("\n  ⚠️  Warning: AUC changed significantly. Check implementation.")
    else:
        brier = brier_score_loss(y_true, p_corrected)
        auc = roc_auc_score(y_true, p_corrected)

        print(f"\nModel Probabilities:")
        print(f"  Brier Score: {brier:.4f}")
        print(f"  AUC:         {auc:.4f}")

    # Plot calibration curves
    print(f"\nComputing calibration curves...")

    fig, axes = plt.subplots(1, 2 if has_uncorrected else 1, figsize=(16 if has_uncorrected else 10, 6))
    if not has_uncorrected:
        axes = [axes]

    if has_uncorrected:
        # Plot uncorrected
        prob_true_unc, prob_pred_unc = calibration_curve(
            y_true, p_uncorrected, n_bins=10, strategy='quantile'
        )

        axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration', linewidth=2)
        axes[0].plot(prob_pred_unc, prob_true_unc, 'o-', 
                    label=f'Uncorrected (Brier={brier_uncorrected:.4f})',
                    linewidth=2, markersize=8, color='red')
        axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
        axes[0].set_ylabel('Fraction of Positives', fontsize=12)
        axes[0].set_title('Calibration Curve - Uncorrected', fontsize=13, fontweight='bold')
        axes[0].legend(loc='upper left', fontsize=10)
        axes[0].grid(alpha=0.3)

        # Plot corrected
        prob_true_cor, prob_pred_cor = calibration_curve(
            y_true, p_corrected, n_bins=10, strategy='quantile'
        )

        axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect calibration', linewidth=2)
        axes[1].plot(prob_pred_cor, prob_true_cor, 'o-',
                    label=f'Corrected (Brier={brier_corrected:.4f})',
                    linewidth=2, markersize=8, color='green')
        axes[1].set_xlabel('Mean Predicted Probability', fontsize=12)
        axes[1].set_ylabel('Fraction of Positives', fontsize=12)
        axes[1].set_title('Calibration Curve - Corrected (King & Zeng)', fontsize=13, fontweight='bold')
        axes[1].legend(loc='upper left', fontsize=10)
        axes[1].grid(alpha=0.3)
    else:
        # Plot single calibration curve
        prob_true, prob_pred = calibration_curve(
            y_true, p_corrected, n_bins=10, strategy='quantile'
        )

        axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration', linewidth=2)
        axes[0].plot(prob_pred, prob_true, 'o-',
                    label=f'Model (Brier={brier:.4f})',
                    linewidth=2, markersize=8, color='blue')
        axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
        axes[0].set_ylabel('Fraction of Positives', fontsize=12)
        axes[0].set_title('Calibration Curve', fontsize=13, fontweight='bold')
        axes[0].legend(loc='upper left', fontsize=10)
        axes[0].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary
    print(f"\n" + "="*60)
    print("SUMMARY")
    print("="*60)

    if has_uncorrected:
        if brier_corrected < brier_uncorrected:
            print("✅ Rare events corrections improved calibration")
            print(f"   Brier score reduced by {brier_uncorrected - brier_corrected:.4f}")
        else:
            print("⚠️  Rare events corrections did not improve calibration")
            print("   Consider reviewing tau parameter or data balance")

        if abs(auc_corrected - auc_uncorrected) < 0.01:
            print("✅ AUC preserved (discrimination unchanged)")
        else:
            print("⚠️  AUC changed - this should not happen with proper corrections")
    else:
        print("Model calibration metrics computed.")
        print("To enable rare events corrections, update training config.")

    print("\n" + "="*80)

except FileNotFoundError as e:
    print(f"\n⚠️  {e}")
    print("\nExpected structure:")
    print(f"  RUN_DIR/bar_size=1m/training/{CV_KIND if 'CV_KIND' in dir() else 'purged_kfold'}/fold_X/preds.parquet")

except Exception as e:
    print(f"\n⚠️  Error: {e}")
    import traceback
    traceback.print_exc()

### 4.6.3.1 🔧 DEMO: Generate Synthetic Trials for PBO Testing (Optional)

**⚠️ DEMO/TESTING ONLY - SKIP THIS SECTION WHEN USING REAL TRAINING DATA**

This section is **OPTIONAL** and is provided only for testing and demonstrating PBO functionality.

**Purpose:**
- Generates synthetic hyperparameter search trials to test PBO analysis
- Useful for validating PBO implementation without running full training
- Demonstrates different overfitting scenarios (none, moderate, severe)

**When to use this:**
- Testing PBO functionality after installing the pipeline
- Understanding how PBO responds to different overfitting patterns
- Quick validation without waiting for real training

**When NOT to use this:**
- When you have real trials from Section 4.6 (Train Models)
- For production analysis or research conclusions
- When actual hyperparameter search has been performed

**Note:** Real trials are automatically tracked during Section 4.6 when you run actual training with hyperparameter search. If you have real trials, **skip this section** and proceed directly to Section 4.6.4.

In [ ]:
# DEMO ONLY: Generate synthetic trials for testing PBO functionality
# ⚠️ SKIP THIS CELL IF YOU HAVE REAL TRIALS FROM ACTUAL TRAINING

import sys
from pathlib import Path

print("\n" + "="*80)
print("DEMO: Generating Synthetic Trials for PBO Testing")
print("="*80)
print("\n⚠️  This is for TESTING/DEMO ONLY. Real trials should come from actual training (Section 4.6).\n")

# Import demo functions
sys.path.insert(0, str(Path(RUN_DIR).parent.parent / 'ml_intraday_v3' / 'training'))
from demo_pbo_with_synthetic_trials import create_demo_trials, analyze_demo_trials

# Create demo trials
# Scenarios: 'no_overfitting', 'moderate_overfitting', 'severe_overfitting'
tracker = create_demo_trials(
    run_dir=str(RUN_DIR),
    scenario='moderate_overfitting'  # Change this to test different scenarios
)

print(f"\n✓ Demo trials created at: {tracker.trials_file}")
print(f"\nTrials saved: {len(tracker.trials)}")
print("\nNext step: Run Section 4.6.4 to analyze these synthetic trials.")
print("\n" + "="*80)
print("⚠️  Remember: These are SYNTHETIC trials for testing only!")
print("For production use, always use real trials from Section 4.6.")
print("="*80 + "\n")

### 4.6.4 Enhanced PBO (Probability of Backtest Overfitting) Analysis

**What is PBO?**

PBO (Probability of Backtest Overfitting) measures the likelihood that your "best" configuration was selected due to overfitting rather than true predictive skill.

**How it works:**
1. For each CPCV path, select the best trial based on in-sample performance on all other paths
2. Measure that trial's out-of-sample rank on the held-out path (lambda value)
3. PBO = fraction of paths where lambda < 0.5 (below median)

**Interpretation:**
- **PBO < 0.3**: Low risk - Configuration appears robust
- **PBO 0.3-0.5**: Moderate risk - Monitor carefully
- **PBO > 0.5**: High risk - Likely overfitting, consider reducing search space

**Reference:** López de Prado, M. (2018). Advances in Financial Machine Learning. Chapter 11.


In [ ]:
# Section 4.6.4: Enhanced PBO Analysis

from ml_intraday_v3.experiments.trial_tracker import TrialTracker
from ml_intraday_v3.experiments.diagnostics import (
    compute_pbo_with_confidence,
    plot_pbo_distribution,
    plot_pbo_with_confidence,
    generate_pbo_report
)
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

print("\n" + "="*80)
print("Section 4.6.4: Enhanced PBO (Probability of Backtest Overfitting) Analysis")
print("="*80 + "\n")

# Check if trials exist
trials_path = RUN_DIR / 'trials' / 'trials.json'

if trials_path.exists():
    print(f"Loading trials from: {trials_path}")
    
    # Load trials
    tracker = TrialTracker(RUN_DIR)
    
    # Display trial summary
    summary = tracker.get_summary_stats()
    print(f"\nTrial Summary:")
    print(f"  - Total trials tracked: {summary['n_trials']}")
    print(f"  - CPCV paths: {summary['n_paths']}")
    print(f"  - Model types: {summary['model_types']}")
    print(f"  - Date range: {summary['date_range']['first']} to {summary['date_range']['last']}")
    
    if summary['n_trials'] < 2:
        print("\n⚠️  Warning: Need at least 2 trials to compute PBO.")
        print("   Track trials during hyperparameter search using TrialTracker.")
    else:
        # Convert to DataFrame
        trials_df = tracker.to_dataframe()
        print(f"\nTrials DataFrame shape: {trials_df.shape}")
        print(f"Columns: {list(trials_df.columns)}")
        
        # Compute PBO with confidence intervals
        print("\nComputing PBO with bootstrap confidence intervals...")
        pbo_result = compute_pbo_with_confidence(
            trials_df=trials_df,
            metric_name='roc_auc',
            higher_is_better=True,
            n_bootstrap=1000,
            confidence_level=0.95,
            random_state=42
        )
        
        if pbo_result['pbo'] is not None:
            # Display results
            pbo = pbo_result['pbo']
            pbo_lower = pbo_result.get('pbo_lower')
            pbo_upper = pbo_result.get('pbo_upper')
            
            print(f"\n{'='*60}")
            print(f"PBO Results:")
            print(f"{'='*60}")
            print(f"PBO = {pbo:.3f} ({pbo*100:.1f}%)")
            if pbo_lower is not None and pbo_upper is not None:
                print(f"95% CI: [{pbo_lower:.3f}, {pbo_upper:.3f}]")
            print(f"\nLambda Statistics:")
            print(f"  - Mean: {pbo_result['lambda_mean']:.3f}")
            print(f"  - Median: {pbo_result['lambda_median']:.3f}")
            print(f"  - Std: {pbo_result['lambda_std']:.3f}")
            print(f"\nTrials: {pbo_result['n_trials']}")
            print(f"CPCV Paths: {pbo_result['n_paths']}")
            
            # Interpretation
            if pbo > 0.5:
                risk = "🔴 HIGH RISK"
                interpretation = "Likely overfitting - Consider reducing search space or increasing sample size"
            elif pbo > 0.3:
                risk = "🟠 MODERATE RISK"
                interpretation = "Some overfitting risk - Monitor performance carefully"
            else:
                risk = "🟢 LOW RISK"
                interpretation = "Configuration appears robust"
            
            print(f"\nRisk Level: {risk}")
            print(f"Interpretation: {interpretation}")
            print(f"{'='*60}\n")
            
            # Create visualizations
            print("Creating PBO visualizations...")
            
            fig, axes = plt.subplots(1, 2, figsize=(16, 5))
            
            # Lambda distribution
            plot_pbo_distribution(
                lambda_values=pbo_result['lambda_values'],
                pbo_value=pbo,
                ax=axes[0]
            )
            
            # PBO with confidence intervals
            plot_pbo_with_confidence(
                pbo_result=pbo_result,
                ax=axes[1]
            )
            
            plt.tight_layout()
            
            # Save figure
            pbo_fig_path = RUN_DIR / 'pbo_analysis.png'
            fig.savefig(pbo_fig_path, dpi=150, bbox_inches='tight')
            print(f"Saved PBO visualization to: {pbo_fig_path}")
            
            plt.show()
            
            # Generate markdown report
            print("\nGenerating PBO report...")
            report_path = RUN_DIR / 'pbo_report.md'
            report = generate_pbo_report(
                pbo_result=pbo_result,
                save_path=str(report_path)
            )
            print(f"Saved PBO report to: {report_path}")
            
            # Display report in notebook
            print("\n" + "="*80)
            print("PBO Report Preview:")
            print("="*80)
            display(Markdown(report))
            
        else:
            reason = pbo_result.get('reason', 'unknown')
            print(f"\n⚠️  Cannot compute PBO: {reason}")
            print("   Ensure trials have IS/OOS metrics for all CPCV paths.")
    
else:
    print(f"\n⚠️  No trials found at: {trials_path}")
    print("\nTo use PBO analysis, track trials during training:")
    print("""\n
from ml_intraday_v3.experiments.trial_tracker import TrialTracker

tracker = TrialTracker(RUN_DIR)

# During hyperparameter search:
for config in all_configs:
    trial_id = tracker.log_trial(
        config=config,
        model_type='logreg',
        hyperparameters=config['model']['params']
    )
    
    # After CPCV evaluation:
    for path_id, (is_metric, oos_metric) in cpcv_results.items():
        tracker.update_path_metrics(trial_id, path_id, is_metric, oos_metric)

tracker.save()
    """)
    print("\nSee ml_intraday_v3/experiments/PBO_ENHANCED_README.md for details.")

print("\n" + "="*80)
print("Section 4.6.4 Complete")
print("="*80 + "\n")


In [ ]:
# Quick Setup for DSR Analysis
# Run this cell if you're jumping directly to DSR analysis

from pathlib import Path

# Set these variables to point to your run directory
# MODIFY THESE VALUES:
run_dir = 'runs/run_20251224_123456'  # ← Change to your actual run directory
bar_size = '1m'  # ← Change to '1m' or '5m'

# Verify the directory exists
if not Path(run_dir).exists():
    print(f"⚠️ WARNING: Directory does not exist: {run_dir}")
    print("")
    print("Available runs:")
    runs_dir = Path('runs')
    if runs_dir.exists():
        for run in sorted(runs_dir.iterdir()):
            if run.is_dir():
                print(f"  - {run.name}")
    else:
        print("  No runs directory found!")
else:
    print(f"✓ Using run directory: {run_dir}")
    print(f"✓ Using bar size: {bar_size}")
    
    # Show what's in the directory
    bar_dir = Path(run_dir) / f"bar_size={bar_size}"
    if bar_dir.exists():
        print(f"✓ Bar directory exists: {bar_dir}")
    else:
        print(f"⚠️ Bar directory does not exist: {bar_dir}")
        print("")
        print("Available bar sizes in run:")
        for item in Path(run_dir).iterdir():
            if item.is_dir() and item.name.startswith('bar_size='):
                print(f"  - {item.name}")

In [ ]:
# DEMO: Generate Sample Trades for DSR Testing
# Run this cell to create sample trades if you don't have real backtest data yet

import numpy as np
import pandas as pd
from pathlib import Path

print("Generating sample trades for DSR demonstration...\n")

# Create output directory
demo_dir = Path(run_dir) / f"bar_size={bar_size}" / "backtest"
demo_dir.mkdir(parents=True, exist_ok=True)

# Generate realistic sample trades
np.random.seed(42)

# Simulate a strategy with:
# - Positive Sharpe ratio (~1.5)
# - Some skewness (occasional large wins)
# - ~1000 trades over ~200 days

n_trades = 1000
n_days = 200

# Generate timestamps
start_date = pd.Timestamp('2024-01-01 09:30:00')
timestamps = pd.date_range(start=start_date, periods=n_trades, freq='2H')

# Generate PnL with positive expectancy and realistic distribution
# Base returns with positive mean
base_returns = np.random.normal(20, 150, n_trades)  # Mean $20, std $150

# Add occasional larger wins (positive skew)
large_wins = np.random.choice([0, 1], size=n_trades, p=[0.95, 0.05])
pnl = base_returns + large_wins * np.random.exponential(200, n_trades)

# Create trades DataFrame
trades_df = pd.DataFrame({
    'trade_id': range(n_trades),
    'timestamp': timestamps,
    'entry_time': timestamps,
    'exit_time': timestamps + pd.Timedelta(minutes=30),
    'pnl': pnl,
    'direction': np.random.choice(['long', 'short'], n_trades),
    'entry_price': 5000 + np.random.normal(0, 50, n_trades),
    'exit_price': 5000 + np.random.normal(0, 50, n_trades),
    'contracts': 1
})

# Save to parquet
output_file = demo_dir / 'trades.parquet'
trades_df.to_parquet(output_file, index=False)

print(f"✓ Generated {n_trades} sample trades")
print(f"✓ Saved to: {output_file}")
print(f"\nSample Trade Statistics:")
print(f"  Total P&L: ${pnl.sum():,.2f}")
print(f"  Mean P&L per trade: ${pnl.mean():.2f}")
print(f"  Std Dev: ${pnl.std():.2f}")
print(f"  Win Rate: {(pnl > 0).mean():.1%}")
print(f"  Sharpe (raw): {pnl.mean() / pnl.std():.4f}")
print(f"\nNow you can run the DSR analysis cell!")

### 4.6.5 Deflated Sharpe Ratio (DSR) Analysis

**What is DSR?**

DSR (Deflated Sharpe Ratio) is a selection-bias-adjusted Sharpe ratio that accounts for:
1. **Selection bias** from testing multiple configurations (N trials)
2. **Non-normality** of returns (skewness and kurtosis)
3. **Statistical uncertainty** in Sharpe ratio estimation

**Formula:**
```
DSR = Φ((SR - SR*) / σ_SR)
```

where:
- `SR` = Observed Sharpe ratio
- `SR*` = Expected maximum Sharpe from N trials (selection bias adjustment)
- `σ_SR` = Standard error of Sharpe (accounts for skewness, kurtosis)
- `Φ` = Standard normal CDF

**Interpretation:**
- **DSR > 0.95**: 🟢 Strong evidence of skill (p < 0.05 equivalent) - High confidence for deployment
- **DSR 0.50-0.95**: 🟠 Moderate evidence - More likely skill than luck, proceed with caution
- **DSR < 0.50**: 🔴 Weak evidence - Likely overfitting or luck - DO NOT deploy

**Why it matters:**
Testing many configurations and selecting the best inflates the Sharpe ratio by chance. DSR corrects for this bias and provides a probabilistic interpretation similar to a p-value.

**Reference:** Bailey & López de Prado (2014). "The Deflated Sharpe Ratio: Correcting for Selection Bias, Backtest Overfitting and Non-Normality." Journal of Portfolio Management.

In [ ]:
# Section 4.6.5: Deflated Sharpe Ratio (DSR) Analysis

from ml_intraday_v3.experiments.trial_tracker import TrialTracker
from ml_intraday_v3.experiments.diagnostics import (
    compute_dsr,
    plot_dsr_distribution,
    plot_dsr_with_confidence,
    generate_dsr_report
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from pathlib import Path

print("\n" + "="*80)
print("Section 4.6.5: Deflated Sharpe Ratio (DSR) Analysis")
print("="*80 + "\n")

# Check if required variables are defined
if 'run_dir' not in locals():
    print("❌ ERROR: 'run_dir' is not defined!")
    print("")
    print("Please set run_dir before running this cell. For example:")
    print("")
    print("    run_dir = 'runs/run_20251224_123456'")
    print("    bar_size = '1m'  # or '5m'")
    print("")
    print("Or run the earlier cells in this notebook to set these variables.")
    raise NameError("run_dir is not defined. Please set it before running this cell.")

if 'bar_size' not in locals():
    print("⚠️ WARNING: 'bar_size' is not defined. Using default: '1m'")
    bar_size = '1m'

# Get number of trials from TrialTracker
try:
    tracker = TrialTracker(run_dir)
    n_trials = len(tracker.trials)
    print(f"✓ Loaded n_trials from TrialTracker: {n_trials}")
except Exception as e:
    print(f"⚠️ Could not load TrialTracker: {e}")
    n_trials = 10  # Conservative default
    print(f"⚠️ Using conservative default: n_trials = {n_trials}")

# Load backtest results
bar_dir = Path(run_dir) / f"bar_size={bar_size}"
backtest_dir = bar_dir / "backtest"

# Try to find trades file
trades_file = None
possible_locations = [
    backtest_dir / "trades.parquet",
    backtest_dir / "trades.csv",
    bar_dir / "trades.parquet",
    bar_dir / "trades.csv",
    # Also check in training directory
    bar_dir / "training" / "cpcv" / "trades.parquet",
]

for possible_file in possible_locations:
    if possible_file.exists():
        trades_file = possible_file
        break

if trades_file is None:
    print("")
    print("⚠️ No backtest trades file found. Skipping DSR analysis.")
    print("")
    print("Searched in:")
    for loc in possible_locations:
        print(f"  - {loc}")
    print("")
    print("Please run the backtest section first or check your file paths.")
    print("")
else:
    print(f"✓ Loading trades from: {trades_file}")
    
    # Load trades
    if str(trades_file).endswith('.parquet'):
        trades_df = pd.read_parquet(trades_file)
    else:
        trades_df = pd.read_csv(trades_file)
    
    print(f"✓ Loaded {len(trades_df)} trades")
    
    # Extract returns (PnL per trade)
    pnl_col = None
    for col in ['pnl', 'net_pnl', 'trade_pnl', 'profit_loss']:
        if col in trades_df.columns:
            pnl_col = col
            break
    
    if pnl_col is None:
        print("")
        print("⚠️ Could not find PnL column in trades data")
        print(f"Available columns: {list(trades_df.columns)}")
        print("")
        returns = None
    else:
        returns = trades_df[pnl_col].values
        print(f"✓ Using column '{pnl_col}' for returns")
    
    if returns is not None and len(returns) > 0:
        # Estimate trades per day for annualization
        time_col = None
        for col in ['timestamp', 'entry_time', 'exit_time', 'time']:
            if col in trades_df.columns:
                time_col = col
                break
        
        if time_col:
            try:
                trades_df[time_col] = pd.to_datetime(trades_df[time_col])
                n_days = (trades_df[time_col].max() - trades_df[time_col].min()).days + 1
                trades_per_day = len(trades_df) / max(n_days, 1)
                annualization_factor = np.sqrt(252 * trades_per_day)
                print(f"✓ Estimated {trades_per_day:.1f} trades/day → annualization factor: {annualization_factor:.2f}")
            except:
                # Default estimate
                trades_per_day = 5
                annualization_factor = np.sqrt(252 * trades_per_day)
                print(f"⚠️ Could not parse timestamps. Using default: {trades_per_day} trades/day → annualization factor: {annualization_factor:.2f}")
        else:
            # Default estimate
            trades_per_day = 5
            annualization_factor = np.sqrt(252 * trades_per_day)
            print(f"⚠️ No timestamp column found. Using default: {trades_per_day} trades/day → annualization factor: {annualization_factor:.2f}")
        
        # Compute DSR
        print("\nComputing DSR...")
        dsr_result = compute_dsr(
            returns=returns,
            n_trials=n_trials,
            target_sharpe=0.0,
            annualization_factor=annualization_factor
        )
        
        if dsr_result.get('dsr') is not None:
            # Display results
            print(f"\n{'='*80}")
            print("DSR Results")
            print(f"{'='*80}\n")
            
            dsr = dsr_result['dsr']
            sharpe = dsr_result['sharpe']
            sharpe_raw = dsr_result['sharpe_raw']
            sr_star = dsr_result['sr_star']
            sr_std = dsr_result['sr_std']
            skew = dsr_result['skewness']
            kurt = dsr_result['kurtosis']
            
            print(f"DSR: {dsr:.4f} ({dsr*100:.1f}%)")
            print(f"\nSharpe Ratio Analysis:")
            print(f"  Observed Sharpe (annualized): {sharpe:.4f}")
            print(f"  Observed Sharpe (raw): {sharpe_raw:.4f}")
            print(f"  Expected Max SR* (from {n_trials} trials): {sr_star:.4f}")
            print(f"  Standard Error (σ_SR): {sr_std:.4f}")
            print(f"  Z-Score: {dsr_result['z_score']:.4f}")
            
            print(f"\nReturn Distribution:")
            print(f"  N observations: {dsr_result['n_obs']:,}")
            print(f"  Skewness: {skew:.4f}")
            print(f"  Kurtosis: {kurt:.4f}")
            
            # Risk assessment
            if dsr > 0.95:
                risk = "🟢 STRONG EVIDENCE"
                interpretation = "Very likely skill, not luck (p < 0.05 equivalent)"
                action = "High confidence for deployment"
            elif dsr > 0.5:
                risk = "🟠 MODERATE EVIDENCE"
                interpretation = "More likely skill than luck, but not conclusive"
                action = "Proceed with caution, monitor closely"
            else:
                risk = "🔴 WEAK EVIDENCE"
                interpretation = "Likely overfitting or luck, not genuine skill"
                action = "DO NOT deploy - likely overfitting"
            
            print(f"\nRisk Assessment:")
            print(f"  Risk Level: {risk}")
            print(f"  Interpretation: {interpretation}")
            print(f"  Recommended Action: {action}")
            
            # Generate report
            dsr_output_dir = bar_dir / "dsr_analysis"
            dsr_output_dir.mkdir(exist_ok=True, parents=True)
            
            report_path = dsr_output_dir / "dsr_report.md"
            generate_dsr_report(dsr_result, save_path=str(report_path))
            print(f"\n✓ DSR report saved to: {report_path}")
            
            # Display report preview in notebook
            print("\n" + "="*80)
            print("DSR Report Preview")
            print("="*80)
            try:
                with open(report_path, 'r') as f:
                    report_content = f.read()
                # Show first part of report
                preview_length = min(2000, len(report_content))
                display(Markdown(report_content[:preview_length] + "\n\n... (see full report in file)"))
            except Exception as e:
                print(f"Could not display report preview: {e}")
            
        else:
            print(f"\n⚠️ DSR computation failed: {dsr_result.get('reason')}")
    else:
        print("\n⚠️ No valid returns found in trades data")

print("\n" + "="*80)
print("DSR Analysis Complete")
print("="*80 + "\n")

### 4.6.6 Enhanced CPCV Analysis

**What is Enhanced CPCV?**

Enhanced Combinatorial Purged Cross-Validation (CPCV) evaluates model performance across multiple validation paths to assess stability and detect overfitting.

**Key Features:**
- **Multiple validation paths**: Tests performance across different train/test splits
- **Path selection strategies**: Lexicographic, balanced (recommended), or random
- **Distribution analysis**: Quartiles, percentiles, IS/OOS ratios
- **Automated validation gates**: Quality checks for deployment readiness
- **Rich visualization**: Box + violin plots showing performance distributions

**Why CPCV Matters:**

A single train/test split can be misleading. CPCV tests your model across many independent validation paths and shows the **distribution** of performance, not just a point estimate.

**Interpretation:**
- **Median OOS Sharpe > 0.8**: Good typical performance
- **Q25 OOS Sharpe > 0.2**: Even worst quartile is acceptable
- **IS/OOS Ratio < 1.5**: Low overfitting (IS not much better than OOS)
- **All gates passed**: Model is deployment-ready

**Reference:** López de Prado (2018). "Advances in Financial Machine Learning." Chapter 7.

In [ ]:
# DEMO: Generate Sample Data for CPCV Testing
# Run this cell if you want to test CPCV without running the full pipeline

import numpy as np
import pandas as pd

print("Generating sample data for CPCV demonstration...\n")

np.random.seed(42)

# Generate sample events (1000 events over time)
n_events = 1000
dates = pd.date_range('2023-01-01', periods=n_events, freq='1h')

events_df = pd.DataFrame({
    'event_id': range(n_events),  # Required by build_purged_kfold_splits
    't0': dates,
    't1': dates + pd.Timedelta(hours=2)  # 2-hour events
})
events_df.index = pd.RangeIndex(n_events)

print(f"✓ Generated {len(events_df)} events")
print(f"  Date range: {events_df['t0'].min()} to {events_df['t0'].max()}")

# Generate sample features (10 features with some signal)
n_features = 10
X = pd.DataFrame(
    np.random.randn(n_events, n_features),
    columns=[f'feature_{i}' for i in range(n_features)]
)
# Add event_id for alignment with CPCV utilities
X['event_id'] = events_df['event_id'].values
X.index = pd.Index(events_df['event_id'], name='event_id')

print(f"✓ Generated feature matrix X with shape {X.shape}")

# Generate labels with some predictable signal
# Signal: label is 1 if feature_0 + feature_1 > 0, else 0
y = pd.Series(
    (X['feature_0'] + X['feature_1'] > 0).astype(int),
    index=X.index,
    name='label'
)

print(f"✓ Generated labels y with {y.sum()} positive samples ({y.mean()*100:.1f}%)")

# Generate sample weights (optional - uniform weights for demo)
sample_weights = pd.Series(
    np.ones(n_events),
    index=X.index,
    name='weight'
)

print(f"✓ Generated sample weights (uniform)")

print("\n" + "="*80)
print("Sample Data Ready for CPCV Analysis")
print("="*80)
print(f"\nData Summary:")
print(f"  Events: {len(events_df)} with columns {list(events_df.columns)}")
print(f"  Features (X): {X.shape[0]} samples × {X.shape[1]} features")
print(f"  Labels (y): {len(y)} samples, {y.sum()} positive")
print(f"  Weights: {len(sample_weights)} samples")

print(f"\nNow you can run the CPCV analysis cell below!")


In [ ]:
# Section 4.6.6: Enhanced CPCV Analysis

from ml_intraday_v3.validation.cpcv import (
    build_cpcv_paths,
    evaluate_cpcv_paths,
    analyze_cpcv_distributions,
    check_cpcv_gates,
    plot_cpcv_performance
)
from ml_intraday_v3.validation import build_purged_kfold_splits
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

print("="*80)
print("Section 4.6.6: Enhanced CPCV Analysis")
print("="*80)

# Check if we have required data
if 'X' in locals() and 'y' in locals() and 'events_df' in locals():
    print("\n✓ Found required data (X, y, events_df)")
    
    # 0. Create bars_index (required for purged CV)
    # Use event times as proxy for bar index
    bars_index = pd.DatetimeIndex(events_df['t0'].values)
    
    # 1. Build base folds using Purged K-Fold
    print("\n" + "─"*80)
    print("Step 0: Building Base Purged K-Fold Splits")
    print("─"*80)
    
    n_splits = 6
    embargo_bars = int(0.01 * len(events_df))  # 1% embargo
    
    base_folds = build_purged_kfold_splits(
        events_df=events_df,
        bars_index=bars_index,
        n_splits=n_splits,
        embargo_bars=embargo_bars
    )
    
    print(f"✓ Built {len(base_folds)} base folds with purging and embargo")
    print(f"  Embargo: {embargo_bars} bars")
    
    # 2. Build CPCV paths from base folds
    print("\n" + "─"*80)
    print("Step 1: Building CPCV Paths")
    print("─"*80)
    
    paths = build_cpcv_paths(
        base_folds=base_folds,
        events_df=events_df,
        bars_index=bars_index,
        K=n_splits,
        test_groups=2,
        embargo_bars=embargo_bars,
        max_paths=15,  # C(6,2) = 15
        selection="balanced",  # Ensures equal fold coverage
        random_state=42
    )
    
    print(f"✓ Built {len(paths)} CPCV paths using balanced selection")
    print(f"  Each path has ~{len(paths[0]['test_event_ids'])} test samples")
    
    # 3. Evaluate performance across all paths
    print("\n" + "─"*80)
    print("Step 2: Evaluating Performance Across Paths")
    print("─"*80)
    
    # Define model factory
    def model_factory():
        return LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        )
    
    # Evaluate with multiple metrics
    perf_df = evaluate_cpcv_paths(
        paths=paths,
        model_factory=model_factory,
        X=X,
        y=y,
        events_df=events_df,
        metrics=['roc_auc', 'accuracy'],
        sample_weight=sample_weights if 'sample_weights' in locals() else None
    )
    
    print(f"✓ Evaluated {len(perf_df)} metric values across {len(paths)} paths")
    print(f"\nPerformance sample (first 10 rows):")
    print(perf_df.head(10).to_string(index=False))
    
    # 4. Analyze distributions
    print("\n" + "─"*80)
    print("Step 3: Distribution Analysis")
    print("─"*80)
    
    stats = analyze_cpcv_distributions(
        perf_df=perf_df,
        metrics=['roc_auc', 'accuracy']
    )
    
    print("\n📊 ROC-AUC Distribution:")
    print(f"  IS  - Median: {stats['roc_auc']['is']['median']:.4f}, "
          f"Q25: {stats['roc_auc']['is']['q25']:.4f}, "
          f"Q75: {stats['roc_auc']['is']['q75']:.4f}")
    print(f"  OOS - Median: {stats['roc_auc']['oos']['median']:.4f}, "
          f"Q25: {stats['roc_auc']['oos']['q25']:.4f}, "
          f"Q75: {stats['roc_auc']['oos']['q75']:.4f}")
    print(f"  IS/OOS Ratio: {stats['roc_auc']['is_oos_ratio']:.3f}")

    print("\n📊 Accuracy Distribution:")
    print(f"  IS  - Median: {stats['accuracy']['is']['median']:.4f}, "
          f"Q25: {stats['accuracy']['is']['q25']:.4f}, "
          f"Q75: {stats['accuracy']['is']['q75']:.4f}")
    print(f"  OOS - Median: {stats['accuracy']['oos']['median']:.4f}, "
          f"Q25: {stats['accuracy']['oos']['q25']:.4f}, "
          f"Q75: {stats['accuracy']['oos']['q75']:.4f}")
    print(f"  IS/OOS Ratio: {stats['accuracy']['is_oos_ratio']:.3f}")
    
    # 5. Check validation gates
    print("\n" + "─"*80)
    print("Step 4: Validation Gates")
    print("─"*80)
    
    # Custom gates for this demo (lenient thresholds)
    # check_cpcv_gates expects scalar thresholds, not nested dicts
    gates_config = {
        'median_roc_auc': 0.55,
        'is_oos_ratio_max': 1.5
    }

    gates_result = check_cpcv_gates(
        perf_df=perf_df,
        stats=stats,
        gates_config=gates_config
    )
    
    print(f"\n🚦 Validation Gate Results:")
    print(f"  Summary: {gates_result.get('summary', 'n/a')}")

    gates_obj = gates_result.get('gates', {})
    passed_count = gates_result.get('passed_count')
    total_count = gates_result.get('total_count')

    # Backward compatible fallback if counts are not present
    if passed_count is None or total_count is None:
        passed_count = sum(1 for g in gates_obj.values() if g.get('passed'))
        total_count = len(gates_obj)

    print(f"  Total gates checked: {total_count}")
    print(f"  ✅ Passed: {passed_count}")
    print(f"  ❌ Failed: {total_count - passed_count}")

    # Show any failed gates with details
    failed_gates = [name for name, g in gates_obj.items() if not g.get('passed')]
    if failed_gates:
        print("  Failed gates:")
        for name in failed_gates:
            g = gates_obj[name]
            print(f"    - {name}: {g.get('message', 'failed')}")
    
    all_passed = gates_result.get('passed')
    gates_failed = gates_result.get('gates_failed', [])

    if all_passed:
        print("\n✅ All validation gates PASSED - Model shows stable performance")
    else:
        print("\n⚠️  Some validation gates FAILED:")
        for failure in gates_failed:
            gate_name = failure.get('gate_name', 'gate')
            actual = failure.get('actual')
            threshold = failure.get('threshold')
            msg = failure.get('message', '')
            if actual is not None and threshold is not None:
                print(f"    - {gate_name}: actual={actual:.3f} vs threshold={threshold:.3f}")
            else:
                print(f"    - {gate_name}: {msg or 'failed'}")
    
    # 6. Visualize
    print("\n" + "─"*80)
    print("Step 5: Visualization")
    print("─"*80)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROC-AUC distribution
    plot_cpcv_performance(
        perf_df=perf_df,
        metric='roc_auc',
        ax=axes[0],
        show_thresholds=True
    )
    axes[0].set_title('ROC-AUC Distribution (IS vs OOS)', fontsize=12, fontweight='bold')
    
    # Accuracy distribution
    plot_cpcv_performance(
        perf_df=perf_df,
        metric='accuracy',
        ax=axes[1],
        show_thresholds=True
    )
    axes[1].set_title('Accuracy Distribution (IS vs OOS)', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ CPCV visualization complete")
    
    # 7. Save results (optional)
    if 'run_dir' in locals() and 'bar_size' in locals():
        from pathlib import Path
        import json
        
        cpcv_dir = Path(run_dir) / f"bar_size={bar_size}" / "cpcv_analysis"
        cpcv_dir.mkdir(parents=True, exist_ok=True)
        
        # Save performance DataFrame
        perf_path = cpcv_dir / "cpcv_performance.csv"
        perf_df.to_csv(perf_path, index=False)
        
        # Save statistics
        stats_path = cpcv_dir / "cpcv_statistics.json"
        with open(stats_path, 'w') as f:
            json.dump(stats, f, indent=2)
        
        # Save gates result
        gates_path = cpcv_dir / "cpcv_gates.json"
        with open(gates_path, 'w') as f:
            json.dump(gates_result, f, indent=2)
        
        print(f"\n📁 Results saved to:")
        print(f"  - Performance: {perf_path}")
        print(f"  - Statistics: {stats_path}")
        print(f"  - Gates: {gates_path}")
    
    print("\n" + "="*80)
    print("Enhanced CPCV Analysis Complete")
    print("="*80)
    
else:
    print("\n⚠️  Required data not found (X, y, events_df)")
    print("Please run the earlier sections of the notebook first to generate:")
    print("  - X: Feature matrix")
    print("  - y: Labels")
    print("  - events_df: Events DataFrame with t0, t1 columns")


### 4.6.7 Regime-Aware Feature Scaling

**What is Regime-Aware Scaling?**

Standard feature scaling uses global statistics (mean, std) across all data. This causes **distribution shift** when market regimes (volatility, trend) differ between train and test.

**Regime-aware scaling** computes separate statistics for each market regime, ensuring features have consistent distributions regardless of which regimes appear in train vs test.

**Key Benefits:**
- **Prevents distribution shift** between train/test
- **Stable performance** across different market conditions
- **Better generalization** when volatility or trend changes

**How It Works:**

1. **Detect Regimes**: Classify market into regimes (e.g., low/medium/high volatility)
2. **Scale Per Regime**: Compute mean/std separately for each regime
3. **Transform**: Scale features using regime-specific statistics

**Example:**
- Low volatility regime: mean=0.5, std=0.1
- High volatility regime: mean=1.5, std=0.5
- Features in each regime are scaled using their own statistics

**Probability Calibration:**

After scaling, we also calibrate model probabilities to ensure reliability:
- **Uncalibrated**: P(predicted=0.9) but actual frequency is only 70%
- **Calibrated**: P(predicted=0.9) and actual frequency is ≈90%

**References:**
- López de Prado (2018). "Advances in Financial Machine Learning." Chapter 19.
- Niculescu-Mizil & Caruana (2005). "Predicting Good Probabilities."


In [ ]:
# Section 4.6.7: Regime-Aware Feature Scaling

from ml_intraday_v3.features.regime_detector import (
    detect_volatility_regime,
    detect_trend_regime,
    detect_combined_regime,
    get_regime_labels
)
from ml_intraday_v3.features.regime_scaler import RegimeAwareScaler
from ml_intraday_v3.features.calibration import (
    calibrate_probabilities,
    evaluate_calibration,
    plot_calibration_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
import numpy as np

print("="*80)
print("Section 4.6.7: Regime-Aware Feature Scaling")
print("="*80)

# Check if we have required data
if 'X' in locals() and 'y' in locals() and 'events_df' in locals():
    print("\n✓ Found required data (X, y, events_df)")
    
    # Generate price series for regime detection (if not exists)
    if 'prices' not in locals():
        # Simulate price series from features
        print("\n📊 Generating price series from features...")
        prices = pd.Series(
            100 + np.cumsum(X.iloc[:, 0].values * 0.01),
            index=X.index
        )
        print(f"✓ Generated price series: {len(prices)} bars")
    
    # 1. Detect Market Regimes
    print("\n" + "─"*80)
    print("Step 1: Detecting Market Regimes")
    print("─"*80)
    
    returns = prices.pct_change().fillna(0)
    
    vol_regime, trend_regime, combined_regime = detect_combined_regime(
        prices=prices,
        returns=returns,
        vol_window=20,
        trend_window=50,
        n_vol_regimes=3,
        n_trend_regimes=3
    )
    
    print(f"✓ Detected regimes for {len(combined_regime)} samples")
    print(f"\n📊 Volatility Regime Distribution:")
    print(vol_regime.value_counts().sort_index())
    print(f"\n📊 Trend Regime Distribution:")
    print(trend_regime.value_counts().sort_index())
    print(f"\n📊 Combined Regime Distribution:")
    print(combined_regime.value_counts().sort_index())
    
    # Get human-readable labels
    labels = get_regime_labels(n_vol_regimes=3, n_trend_regimes=3)
    print(f"\n📋 Regime Labels:")
    for regime_id in sorted(combined_regime.unique()):
        count = (combined_regime == regime_id).sum()
        pct = 100 * count / len(combined_regime)
        print(f"  {regime_id}: {labels[regime_id]:<25} ({count:>4} samples, {pct:>5.1f}%)")
    
    # 2. Compare Standard vs Regime-Aware Scaling
    print("\n" + "─"*80)
    print("Step 2: Comparing Scaling Methods")
    print("─"*80)
    
    # Split data (80/20 train/test, no shuffle for time series)
    train_size = int(0.8 * len(X))
    
    X_train = X.iloc[:train_size]
    X_test = X.iloc[train_size:]
    y_train = y.iloc[:train_size]
    y_test = y.iloc[train_size:]
    regime_train = combined_regime.iloc[:train_size]
    regime_test = combined_regime.iloc[train_size:]
    
    print(f"Train: {len(X_train)} samples")
    print(f"Test:  {len(X_test)} samples")
    
    # 2a. Standard Scaling (baseline)
    from sklearn.preprocessing import StandardScaler
    
    standard_scaler = StandardScaler()
    X_train_standard = standard_scaler.fit_transform(X_train)
    X_test_standard = standard_scaler.transform(X_test)
    
    # 2b. Regime-Aware Scaling
    regime_scaler = RegimeAwareScaler(min_samples_per_regime=10, fallback_to_global=True)
    X_train_regime = regime_scaler.fit_transform(X_train.values, regime_train.values)
    X_test_regime = regime_scaler.transform(X_test.values, regime_test.values)
    
    print(f"\n✓ Applied both scaling methods")
    
    # Inspect regime-specific statistics
    stats = regime_scaler.get_regime_stats()
    print(f"\n📊 Regime-Aware Scaler Statistics:")
    print(f"  Regimes fitted: {stats['regimes_seen']}")
    print(f"  Features: {stats['n_features']}")
    
    # 3. Train Models and Compare
    print("\n" + "─"*80)
    print("Step 3: Training Models")
    print("─"*80)
    
    # Model with standard scaling
    model_standard = LogisticRegression(max_iter=1000, random_state=42)
    model_standard.fit(X_train_standard, y_train)
    y_prob_standard = model_standard.predict_proba(X_test_standard)[:, 1]
    
    # Model with regime-aware scaling
    model_regime = LogisticRegression(max_iter=1000, random_state=42)
    model_regime.fit(X_train_regime, y_train)
    y_prob_regime = model_regime.predict_proba(X_test_regime)[:, 1]
    
    # Compare performance
    auc_standard = roc_auc_score(y_test, y_prob_standard)
    auc_regime = roc_auc_score(y_test, y_prob_regime)
    
    acc_standard = accuracy_score(y_test, y_prob_standard > 0.5)
    acc_regime = accuracy_score(y_test, y_prob_regime > 0.5)
    
    print(f"\n📊 Model Performance Comparison:")
    print(f"  Standard Scaling:")
    print(f"    ROC-AUC: {auc_standard:.4f}")
    print(f"    Accuracy: {acc_standard:.4f}")
    print(f"  Regime-Aware Scaling:")
    print(f"    ROC-AUC: {auc_regime:.4f}")
    print(f"    Accuracy: {acc_regime:.4f}")
    print(f"  Improvement: {(auc_regime - auc_standard):.4f} AUC")
    
    # 4. Probability Calibration
    print("\n" + "─"*80)
    print("Step 4: Probability Calibration")
    print("─"*80)
    
    # Calibrate regime-aware model probabilities
    y_prob_calibrated, calibrator = calibrate_probabilities(
        y_prob=y_prob_regime,
        y_true=y_test.values,
        method="isotonic",
        return_calibrator=True
    )
    
    # Evaluate calibration
    metrics_before = evaluate_calibration(y_prob_regime, y_test.values, n_bins=10)
    metrics_after = evaluate_calibration(y_prob_calibrated, y_test.values, n_bins=10)
    
    print(f"\n📊 Calibration Metrics:")
    print(f"  Before Calibration:")
    print(f"    Brier Score: {metrics_before['brier_score']:.4f}")
    print(f"    ECE (Expected Calibration Error): {metrics_before['ece']:.4f}")
    print(f"    MCE (Max Calibration Error): {metrics_before['mce']:.4f}")
    print(f"  After Calibration:")
    print(f"    Brier Score: {metrics_after['brier_score']:.4f}")
    print(f"    ECE: {metrics_after['ece']:.4f}")
    print(f"    MCE: {metrics_after['mce']:.4f}")
    print(f"  Improvement:")
    print(f"    ECE: {metrics_before['ece'] - metrics_after['ece']:.4f} (lower is better)")
    
    # 5. Visualizations
    print("\n" + "─"*80)
    print("Step 5: Visualizations")
    print("─"*80)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Plot 1: Regime Distribution Over Time
    ax1 = axes[0, 0]
    combined_regime.plot(ax=ax1, style='.', alpha=0.5, markersize=2)
    ax1.set_ylabel('Regime ID', fontsize=11)
    ax1.set_xlabel('Time', fontsize=11)
    ax1.set_title('Combined Regime Over Time', fontsize=12, fontweight='bold')
    ax1.grid(alpha=0.3)
    
    # Plot 2: Feature Distribution by Regime (first feature)
    ax2 = axes[0, 1]
    for regime_id in sorted(combined_regime.unique())[:5]:  # Show first 5 regimes
        mask = (combined_regime == regime_id).values
        if mask.sum() > 0:
            ax2.hist(
                X.iloc[:, 0].values[mask],
                bins=30,
                alpha=0.5,
                label=f'{labels.get(regime_id, f"Regime {regime_id}")}',
                density=True
            )
    ax2.set_xlabel('Feature Value', fontsize=11)
    ax2.set_ylabel('Density', fontsize=11)
    ax2.set_title('Feature Distribution by Regime', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    
    # Plot 3: Calibration Curve - Before
    plot_calibration_curve(
        y_prob=y_prob_regime,
        y_true=y_test.values,
        n_bins=10,
        ax=axes[1, 0],
        label="Before Calibration"
    )
    axes[1, 0].set_title('Calibration: Before', fontsize=12, fontweight='bold')
    
    # Plot 4: Calibration Curve - After
    plot_calibration_curve(
        y_prob=y_prob_calibrated,
        y_true=y_test.values,
        n_bins=10,
        ax=axes[1, 1],
        label="After Calibration"
    )
    axes[1, 1].set_title('Calibration: After', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualizations complete")
    
    # 6. Summary
    print("\n" + "="*80)
    print("Regime-Aware Feature Scaling Complete")
    print("="*80)
    
    print(f"\n📋 Summary:")
    print(f"  ✓ Detected {len(combined_regime.unique())} unique regimes")
    print(f"  ✓ Regime-aware scaling {'improved' if auc_regime > auc_standard else 'matched'} performance")
    print(f"  ✓ Probability calibration reduced ECE by {metrics_before['ece'] - metrics_after['ece']:.4f}")
    print(f"\n💡 Key Insight:")
    if auc_regime > auc_standard:
        print(f"  Regime-aware scaling provided {(auc_regime - auc_standard)*100:.2f}% AUC improvement")
        print(f"  by preventing distribution shift between regimes!")
    else:
        print(f"  Both methods performed similarly. Regime-aware scaling provides")
        print(f"  stability guarantee across different market conditions.")
    
else:
    print("\n⚠️  Required data not found (X, y, events_df)")
    print("Please run the earlier sections of the notebook first to generate:")
    print("  - X: Feature matrix")
    print("  - y: Labels")
    print("  - events_df: Events DataFrame")
    print("\nOr run the DEMO data generator cell (Cell 37) to create sample data.")


### 4.7 Run Backtest

In [ ]:
run_cmd([
    "python", "-m", CLI_MODULE, "build-backtest",
    "--run-dir", str(RUN_DIR),
    "--training-dir", str(RUN_DIR),
    "--backtest-config", str(BACKTEST_YAML),
    "--execution-spec", str(EXECUTION_SPEC_YAML),
    "--risk-config", str(RISK_YAML),
    "--cv-kind", CV_KIND,
])

# Analyze backtest
bt_summary, trades_df = analyze_backtest("1m", CV_KIND)

# Overfitting diagnostics (DSR)
of_results = analyze_overfitting(trades_df, "1m")

### 4.7.1 Backtest Performance - Cost Curve Analysis

Analyze backtest results using cost curves to understand:
- How well the model separates profitable vs unprofitable trades
- Optimal risk/reward ratio for the strategy
- Performance across different cost assumptions

In [ ]:
# Backtest Cost Curve Analysis
from ml_intraday_v3.analysis.cost_curves import (
    compute_cost_curve,
    plot_cost_curve,
    compute_trading_cost_curve,
    compute_area_under_cost_curve
)
import pandas as pd
import numpy as np
from pathlib import Path

print("\n" + "="*80)
print("BACKTEST COST CURVE ANALYSIS")
print("="*80)

try:
    bar_size_dir = BAR_SIZE_1M_DIR
    backtest_dir = RUN_DIR / bar_size_dir / "backtests" / CV_KIND
    
    if not backtest_dir.exists():
        raise FileNotFoundError(f"Backtest directory not found: {backtest_dir}")
    
    # Load backtest trades from all folds
    fold_dirs = sorted([d for d in backtest_dir.iterdir() if d.is_dir()])
    
    print(f"\nLoading backtest trades from {len(fold_dirs)} folds...")
    all_trades = []
    
    for fold_dir in fold_dirs:
        trades_file = fold_dir / "trades.parquet"
        if trades_file.exists():
            fold_trades = pd.read_parquet(trades_file)
            all_trades.append(fold_trades)
            print(f"  {fold_dir.name}: {len(fold_trades)} trades")
    
    if not all_trades:
        raise FileNotFoundError(f"No trades.parquet files found in {backtest_dir}")
    
    # Combine all trades
    trades_df = pd.concat(all_trades, ignore_index=True)
    
    print(f"\nTotal trades: {len(trades_df)}")
    print(f"Columns: {list(trades_df.columns)}")
    
    # Filter to executed trades only (have actual P&L)
    if 'executed' in trades_df.columns:
        trades_df = trades_df[trades_df['executed'] == True].copy()
        print(f"Executed trades: {len(trades_df)}")
    
    # Filter trades with probabilities
    if 'p_primary' in trades_df.columns:
        trades_df = trades_df[trades_df['p_primary'].notna()].copy()
        print(f"Trades with probabilities: {len(trades_df)}")
    
    if len(trades_df) == 0:
        raise ValueError("No executed trades with probabilities found")
    
    # Extract labels and probabilities
    # Use pnl_usd column
    if 'pnl_usd' in trades_df.columns:
        y_true_bt = (trades_df['pnl_usd'] > 0).astype(int).values
        y_prob_bt = trades_df['p_primary'].values
        
        print(f"\nTrade Statistics:")
        print(f"  Profitable trades: {y_true_bt.sum()} ({y_true_bt.mean()*100:.1f}%)")
        print(f"  Unprofitable trades: {(1-y_true_bt).sum()} ({(1-y_true_bt.mean())*100:.1f}%)")
        print(f"  Mean P&L: ${trades_df['pnl_usd'].mean():.2f}")
        print(f"  Total P&L: ${trades_df['pnl_usd'].sum():.2f}")
        
        # Compute cost curve
        print("\nComputing backtest cost curve...")
        bt_curve = compute_cost_curve(y_true_bt, y_prob_bt)
        bt_aucc = compute_area_under_cost_curve(bt_curve)
        
        print(f"  AUCC: {bt_aucc:.4f} (lower = better trade selection)")
        
        # Trading-specific analysis
        bt_trading_curve = compute_trading_cost_curve(
            y_true_bt, y_prob_bt,
            risk_reward_ratios=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
        )
        
        optimal_idx = bt_trading_curve['nc'].idxmin()
        optimal_row = bt_trading_curve.loc[optimal_idx]
        
        print(f"\nOptimal Trading Configuration:")
        print(f"  Risk/Reward Ratio: {optimal_row['risk_reward_ratio']:.2f}")
        print(f"  Decision Threshold: {optimal_row['threshold']:.3f}")
        print(f"  Expected TPR (catch profitable): {optimal_row['tpr']:.1%}")
        print(f"  Expected FPR (take unprofitable): {optimal_row['fpr']:.1%}")
        print(f"  Normalized Cost: {optimal_row['nc']:.4f}")
        
        # Visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # Plot 1: Cost curve
        plot_cost_curve(bt_curve, ax=ax1, label="Backtest Performance", color='purple')
        ax1.set_title('Backtest Cost Curve', fontsize=13, fontweight='bold')
        
        # Plot 2: NC vs RR ratio
        ax2.plot(bt_trading_curve['risk_reward_ratio'], bt_trading_curve['nc'], 
                marker='o', linewidth=2, markersize=8, color='purple')
        ax2.axvline(optimal_row['risk_reward_ratio'], color='red', 
                   linestyle='--', alpha=0.5, label=f"Optimal RR={optimal_row['risk_reward_ratio']:.1f}")
        ax2.set_xlabel('Risk/Reward Ratio', fontsize=11)
        ax2.set_ylabel('Normalized Expected Cost (NC)', fontsize=11)
        ax2.set_title('Performance vs Risk/Reward Ratio', fontsize=13, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Summary table
        print("\nPerformance by Risk/Reward Ratio:")
        summary = bt_trading_curve[['risk_reward_ratio', 'nc', 'threshold', 'tpr', 'fpr']].copy()
        summary.columns = ['RR', 'NC', 'Threshold', 'TPR', 'FPR']
        print(summary.to_string(index=False))
        
    else:
        raise ValueError("'pnl_usd' column not found in trades")
        
except FileNotFoundError as e:
    print(f"\n⚠️  {e}")
    print(f"\nExpected structure:")
    print(f"  RUN_DIR/bar_size=1m/backtests/{CV_KIND if 'CV_KIND' in dir() else 'purged_kfold'}/fold_X/trades.parquet")
    
except Exception as e:
    print(f"\n⚠️  Error: {e}")
    print(f"\nDebug info:")
    print(f"  RUN_DIR: {RUN_DIR if 'RUN_DIR' in dir() else 'NOT SET'}")
    import traceback
    traceback.print_exc()


### 4.8 Walk-Forward

In [ ]:
run_cmd([
    "python", "-m", CLI_MODULE, "run-walkforward",
    "--run-dir", str(RUN_DIR),
    "--walkforward-config", str(WALKFORWARD_YAML),
])

# Analyze walk-forward
wf_summary = analyze_walk_forward("1m")

## 5) Monte Carlo Simulations

### 5.1 Trade Sequence Randomization

In [ ]:
# Run Monte Carlo bootstrap on trade sequences
mc_results = monte_carlo_trade_sequence(trades_df, n_simulations=1000, seed=SEED)

### 5.2 Equity Curve Uncertainty

In [ ]:
# Visualize equity curve cone of uncertainty
monte_carlo_equity_curve(trades_df, n_simulations=100, seed=SEED)

### 5.3 PBO Validation Across Multiple Runs

This section demonstrates how to compare PBO across different hyperparameter searches or model variants, and provides guidelines for stopping hyperparameter search based on PBO thresholds.

**Key Questions:**
1. How does PBO change with number of trials (selection bias)?
2. When should we stop hyperparameter search?
3. How do different model variants compare in terms of overfitting risk?

**Guidelines:**
- **Stop if PBO > 0.5**: High overfitting risk, reduce search space
- **Stop if PBO increases**: More trials without improvement suggest overfitting
- **Compare across runs**: Lower PBO is better (less selection bias)


In [ ]:
# Section 5.3: PBO Validation Across Multiple Runs

from ml_intraday_v3.experiments.trial_tracker import TrialTracker
from ml_intraday_v3.experiments.diagnostics import compute_pbo_enhanced
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("\n" + "="*80)
print("Section 5.3: PBO Validation Across Multiple Runs")
print("="*80 + "\n")

trials_path = RUN_DIR / 'trials' / 'trials.json'

if trials_path.exists():
    # Load trials
    tracker = TrialTracker(RUN_DIR)
    trials_df = tracker.to_dataframe()
    
    n_trials = len(trials_df)
    
    if n_trials >= 5:
        print("Analyzing how PBO changes with number of trials...\n")
        
        # Compute PBO for increasing number of trials
        trial_counts = []
        pbo_values = []
        
        # Start with at least 2 trials, then increase
        trial_range = range(2, n_trials + 1)
        
        for n in trial_range:
            subset_df = trials_df.iloc[:n].copy()
            result = compute_pbo_enhanced(
                subset_df,
                metric_name='roc_auc',
                higher_is_better=True
            )
            
            if result['pbo'] is not None:
                trial_counts.append(n)
                pbo_values.append(result['pbo'])
        
        if len(pbo_values) > 0:
            # Plot PBO vs number of trials
            fig, ax = plt.subplots(figsize=(12, 6))
            
            ax.plot(
                trial_counts,
                pbo_values,
                marker='o',
                linewidth=2,
                markersize=8,
                color='steelblue',
                label='PBO vs Trials'
            )
            
            # Reference lines
            ax.axhline(
                0.5,
                color='red',
                linestyle='--',
                linewidth=2,
                alpha=0.7,
                label='High Risk Threshold (0.5)'
            )
            ax.axhline(
                0.3,
                color='orange',
                linestyle='--',
                linewidth=2,
                alpha=0.7,
                label='Moderate Risk Threshold (0.3)'
            )
            
            ax.set_xlabel('Number of Trials', fontsize=12)
            ax.set_ylabel('PBO (Probability of Backtest Overfitting)', fontsize=12)
            ax.set_title(
                'PBO vs Number of Trials (Selection Bias Demonstration)',
                fontsize=14,
                fontweight='bold'
            )
            ax.legend(loc='best', fontsize=10)
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1)
            
            plt.tight_layout()
            
            # Save figure
            pbo_trend_path = RUN_DIR / 'pbo_vs_trials.png'
            fig.savefig(pbo_trend_path, dpi=150, bbox_inches='tight')
            print(f"Saved PBO trend plot to: {pbo_trend_path}")
            
            plt.show()
            
            # Analysis
            print(f"\n{'='*60}")
            print("PBO Trend Analysis:")
            print(f"{'='*60}")
            print(f"Starting PBO (n={trial_counts[0]}): {pbo_values[0]:.3f}")
            print(f"Final PBO (n={trial_counts[-1]}): {pbo_values[-1]:.3f}")
            print(f"Change: {pbo_values[-1] - pbo_values[0]:+.3f}")
            
            # Check if PBO increased
            if pbo_values[-1] > pbo_values[0]:
                print("\n⚠️  Warning: PBO increased with more trials (selection bias)")
                print("   Consider stopping hyperparameter search.")
            else:
                print("\n✓ PBO did not increase significantly with more trials.")
            
            # Stopping criteria
            print(f"\n{'='*60}")
            print("Stopping Criteria Recommendations:")
            print(f"{'='*60}")
            
            final_pbo = pbo_values[-1]
            if final_pbo > 0.5:
                print("🔴 STOP: PBO > 0.5 (high overfitting risk)")
                print("   Actions:")
                print("   1. Reduce hyperparameter search space")
                print("   2. Increase training sample size")
                print("   3. Use simpler models")
                print("   4. Consider ensemble methods instead of single best config")
            elif final_pbo > 0.3:
                print("🟠 CAUTION: PBO > 0.3 (moderate risk)")
                print("   Actions:")
                print("   1. Validate on additional out-of-sample data")
                print("   2. Monitor performance closely after deployment")
                print("   3. Be conservative with position sizing initially")
            else:
                print("🟢 PROCEED: PBO < 0.3 (low risk)")
                print("   Configuration appears robust, but continue monitoring.")
            
            print(f"{'='*60}\n")
            
            # Create summary table
            summary_df = pd.DataFrame({
                'n_trials': trial_counts,
                'pbo': pbo_values,
                'risk_level': [
                    'High' if pbo > 0.5 else 'Moderate' if pbo > 0.3 else 'Low'
                    for pbo in pbo_values
                ]
            })
            
            print("\nPBO Summary Table:")
            print(summary_df.to_string(index=False))
            
            # Save summary
            summary_path = RUN_DIR / 'pbo_summary.csv'
            summary_df.to_csv(summary_path, index=False)
            print(f"\nSaved summary to: {summary_path}")
        
        else:
            print("Could not compute PBO for trial subsets.")
    
    else:
        print(f"Need at least 5 trials for trend analysis (have {n_trials}).")
        print("This section demonstrates how PBO changes with number of trials.")
    
    # Compare model types if multiple exist
    print("\n" + "="*80)
    print("Model Type Comparison:")
    print("="*80 + "\n")
    
    if 'model_type' in trials_df.columns:
        model_types = trials_df['model_type'].unique()
        
        if len(model_types) > 1:
            print(f"Found {len(model_types)} model types: {list(model_types)}\n")
            
            model_pbo_comparison = []
            
            for model_type in model_types:
                model_df = trials_df[trials_df['model_type'] == model_type].copy()
                
                if len(model_df) >= 2:
                    result = compute_pbo_enhanced(
                        model_df,
                        metric_name='roc_auc',
                        higher_is_better=True
                    )
                    
                    if result['pbo'] is not None:
                        model_pbo_comparison.append({
                            'model_type': model_type,
                            'n_trials': len(model_df),
                            'pbo': result['pbo'],
                            'lambda_mean': result['lambda_mean']
                        })
            
            if model_pbo_comparison:
                comparison_df = pd.DataFrame(model_pbo_comparison)
                comparison_df = comparison_df.sort_values('pbo')
                
                print("Model Type PBO Comparison:")
                print(comparison_df.to_string(index=False))
                print(f"\nBest model type (lowest PBO): {comparison_df.iloc[0]['model_type']}")
        else:
            print(f"Only one model type found: {model_types[0]}")
    else:
        print("Model type information not available in trials.")

else:
    print(f"No trials found at: {trials_path}")
    print("Run Section 4.6.4 first and ensure trials are tracked during training.")

print("\n" + "="*80)
print("Section 5.3 Complete")
print("="*80 + "\n")


## 5.5) Model Comparison - Cost Curves

Compare multiple model variants or strategies using cost curves.

This section demonstrates:
- Multi-model comparison on same dataset
- AUCC ranking (lower is better)
- Cost curve difference visualization
- Dominance analysis across cost ratios

In [ ]:
# Model Comparison with Cost Curves
from ml_intraday_v3.analysis.cost_curves import (
    compare_models_cost_curves,
    plot_cost_difference,
    compute_cost_curve,
    compute_area_under_cost_curve
)
import pandas as pd
import numpy as np
from pathlib import Path

print("\n" + "="*80)
print("MODEL COMPARISON - COST CURVES")
print("="*80)

try:
    bar_size_dir = BAR_SIZE_1M_DIR
    training_dir = RUN_DIR / bar_size_dir / "training" / CV_KIND
    
    # Load primary model predictions
    fold_dirs = sorted([d for d in training_dir.iterdir() if d.is_dir()])
    
    if not fold_dirs:
        raise FileNotFoundError(f"No fold directories found in {training_dir}")
    
    print(f"\nLoading predictions from {len(fold_dirs)} folds...")
    all_preds = []
    
    for fold_dir in fold_dirs:
        preds_file = fold_dir / "preds.parquet"
        if preds_file.exists():
            all_preds.append(pd.read_parquet(preds_file))
    
    if not all_preds:
        raise FileNotFoundError(f"No preds.parquet found in {training_dir}")
    
    preds_df = pd.concat(all_preds, ignore_index=True)
    
    # Extract for primary model
    y_test = (preds_df['y_true'] == 1).astype(int).values
    y_pred_proba = preds_df['p_target'].values
    
    models_to_compare = {
        'Primary Model': (y_test, y_pred_proba)
    }
    
    print(f"  Primary Model: {len(y_test)} predictions")
    
    # Check for meta-labeling (if available)
    if 'p_meta' in preds_df.columns:
        p_meta = preds_df['p_meta'].dropna()
        if len(p_meta) > 0:
            models_to_compare['Meta Model'] = (y_test[:len(p_meta)], p_meta.values)
            print(f"  Meta Model: {len(p_meta)} predictions")
    
    # Create baseline comparison
    if len(models_to_compare) == 1:
        print("  Creating baseline (class prior) for comparison")
        baseline_prob = np.full_like(y_pred_proba, y_test.mean())
        models_to_compare['Baseline (Prior)'] = (y_test, baseline_prob)
    
    print(f"\nComparing {len(models_to_compare)} models...")
    
    # Compare
    fig, curves = compare_models_cost_curves(
        models_to_compare,
        show_confidence=False
    )
    plt.show()
    
    # AUCC comparison
    print("\n" + "-"*80)
    print("AUCC Comparison (Area Under Cost Curve - lower is better):")
    print("-"*80)
    
    aucc_results = []
    for model_name, curve in curves.items():
        aucc = compute_area_under_cost_curve(curve)
        aucc_results.append((model_name, aucc))
    
    aucc_results.sort(key=lambda x: x[1])
    
    for rank, (model_name, aucc) in enumerate(aucc_results, 1):
        star = "⭐" if rank == 1 else "  "
        print(f"{star} {rank}. {model_name:20s}: AUCC = {aucc:.4f}")
    
    best_model = aucc_results[0][0]
    print(f"\n✓ Best model: {best_model}")
    
    # Difference plot if 2 models
    if len(curves) == 2:
        print("\nPlotting cost curve difference...")
        model_names = list(curves.keys())
        
        fig, ax = plt.subplots(figsize=(10, 6))
        plot_cost_difference(
            curves[model_names[0]],
            curves[model_names[1]],
            ax=ax,
            label1=model_names[0],
            label2=model_names[1]
        )
        plt.show()
        
        nc_diff = curves[model_names[0]]['nc'].values - curves[model_names[1]]['nc'].values
        pct_better = (nc_diff < 0).mean() * 100
        print(f"\n{model_names[0]} outperforms {model_names[1]} for {pct_better:.1f}% of cost ratios")
        
except FileNotFoundError as e:
    print(f"\n⚠️  {e}")
    print(f"\nExpected: RUN_DIR/bar_size=1m/training/{CV_KIND if 'CV_KIND' in dir() else 'purged_kfold'}/fold_X/preds.parquet")
    
except Exception as e:
    print(f"\n⚠️  Error: {e}")
    import traceback
    traceback.print_exc()


## 6) Final Summary Report

In [ ]:
print_section("FINAL PIPELINE HEALTH SUMMARY")

# Collect key metrics
if 'train_summary' in locals():
    mm = train_summary.get('metrics_mean', {})
    primary_auc = mm.get('roc_auc', mm.get('roc_auc_target_vs_rest'))
else:
    primary_auc = None

if 'bt_summary' in locals():
    total_trades = sum(s['trades_count'] for s in bt_summary['metrics_by_split'])
    total_pnl = sum(s.get('total_pnl_usd', 0) for s in bt_summary['metrics_by_split'])
else:
    total_trades = 0
    total_pnl = 0

if 'mc_results' in locals() and mc_results:
    prob_profit = (mc_results['total_pnl'] > 0).mean() * 100
else:
    prob_profit = None

# Print summary
print("\nKEY METRICS:")
print("-" * 80)
if primary_auc is not None:
    print(f"  Primary Model AUC (target-vs-rest): {primary_auc:.4f}", end="")
    if primary_auc < 0.52:
        print(" ❌ FAILED (random)")
    elif primary_auc < 0.55:
        print(" ⚠️  WEAK")
    elif primary_auc < 0.60:
        print(" ⚡ MODERATE")
    else:
        print(" ✅ GOOD")

print(f"  Backtest Trades:            {total_trades:,}")
print(f"  Backtest Total PnL:         ${total_pnl:,.0f}", end="")
if total_pnl < 0:
    print(" ❌ LOSING")
elif total_pnl < 1000:
    print(" ⚠️  MARGINAL")
else:
    print(" ✅ PROFITABLE")

if prob_profit is not None:
    print(f"  MC Probability of Profit:   {prob_profit:.1f}%", end="")
    if prob_profit < 50:
        print(" ❌ <50%")
    elif prob_profit < 70:
        print(" ⚠️  MODERATE")
    else:
        print(" ✅ HIGH")

print("-" * 80)

# Overall assessment
print("\nOVERALL ASSESSMENT:")
print("-" * 80)
if primary_auc and primary_auc < 0.52 and total_pnl < 0:
    print("🔴 CRITICAL: Strategy FAILED on all metrics")
    print("   → DO NOT DEPLOY to live trading")
    print("   → Revisit labeling scheme and feature engineering")
elif primary_auc and primary_auc < 0.55:
    print("🟠 WARNING: Weak predictive power")
    print("   → Consider improving features before deployment")
elif total_pnl < 0:
    print("🟠 WARNING: Losing money in backtest")
    print("   → Review execution costs and risk management")
else:
    print("🟢 PASSED: Strategy shows some potential")
    print("   → Continue with robustness testing and walk-forward validation")
print("-" * 80)

print("\n✅ ANALYSIS COMPLETE")